# Pregunta 3 — Monitoreo de fauna silvestre con cámaras trampa (YOLO)
**Redes Neuronales y Aprendizaje Profundo · Examen parcial 2026-II · UNI FIIS**

Dataset: **Caltech Camera Traps – subconjunto de referencia CCT20** (Beery, Van Horn y Perona, *Recognition in Terra Incognita*, ECCV 2018), distribuido por LILA BC. Se usan sus particiones oficiales:

| Partición | Ubicaciones de cámara | Uso en este notebook |
|---|---|---|
| `train` | "vistas" | entrenamiento |
| `cis_val` + `trans_val` | vistas + nuevas | selección del mejor checkpoint |
| `cis_test` | **vistas** en entrenamiento | evaluación "en distribución" |
| `trans_test` | **nunca vistas** | evaluación de generalización |

En LILA, `eccv_train` = ubicaciones *train/cis* y `eccv_val` = ubicaciones *trans*; por eso la comparación **cis vs trans** es exactamente el experimento entre ubicaciones que pide el examen.

### Mapa entregable → sección
| Entregable (P3) | Sección |
|---|---|
| 1. YOLO con decisiones documentadas (resolución, aumento, vacías como fondo) | 1, 6, 7 |
| 2. mAP@0.5 y mAP@0.5:0.95 por especie + efecto del desbalance | 8, 9 |
| 3. Generalización entre ubicaciones + evidencia de aprendizaje del fondo (Grad-CAM) | 10, 11 |
| 4. Robustez ante oclusión y tamaño del objeto | 12 |
| 5. Módulo de monitoreo ecológico con demostración en cámara no vista | 13 |
| Extra para el informe: ablación (vacías / *Repeat Factor Sampling*), velocidad | 14, 15 |
| Exportación de todo en un `.zip` | 16 |

### Cómo usarlo en Colab
1. **Entorno de ejecución → Cambiar tipo de entorno → GPU** (T4 o superior).
2. Revisa la celda de configuración (sección 1) y luego **Entorno de ejecución → Ejecutar todo**.
3. Todo lo costoso se guarda en Google Drive (`MyDrive/P3_CCT_YOLO`): pesos, predicciones y el dataset ya convertido. **Si Colab se desconecta, vuelve a "Ejecutar todo"**: el entrenamiento se reanuda desde `last.pt` y lo terminado se salta.
4. Al final se genera y descarga `P3_resultados_<fecha>.zip` con tablas (CSV), figuras (PNG), métricas (`resumen.json`) y un `RESUMEN.md` legible.

Tiempo orientativo con perfil `estandar` en una T4: preparación de datos 25–35 min, entrenamiento principal 2–3 h, ablaciones 3 × ~1 h, análisis 30–40 min.

In [ ]:
#@title 1. Configuración del experimento
import os, sys, json, time, math, random, shutil, re, subprocess, platform, zipfile, tarfile, warnings
import datetime as dt
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

# True = prueba rápida de punta a punta con datos sintéticos (no descarga nada). Útil para validar el código.
SMOKE_TEST = os.environ.get("P3_SMOKE_TEST", "0") == "1"

PROFILE = "estandar"   # "rapido" | "estandar" | "completo"
PROFILES = {
    "rapido":   dict(MODEL_WEIGHTS="yolov8n.pt", IMGSZ=512, EPOCHS=20, BATCH=32),
    "estandar": dict(MODEL_WEIGHTS="yolov8s.pt", IMGSZ=640, EPOCHS=30, BATCH=16),
    "completo": dict(MODEL_WEIGHTS="yolov8m.pt", IMGSZ=640, EPOCHS=50, BATCH=16),
}
MODEL_WEIGHTS = PROFILES[PROFILE]["MODEL_WEIGHTS"]   # backbone preentrenado en COCO
IMGSZ         = PROFILES[PROFILE]["IMGSZ"]            # resolución de entrada (lado mayor)
EPOCHS        = PROFILES[PROFILE]["EPOCHS"]
BATCH         = PROFILES[PROFILE]["BATCH"]
PATIENCE      = 10        # early stopping sobre mAP50-95 de validación
SEED          = 42

# --- Imágenes vacías (fondo) ---
BG_RATIO            = 0.10    # fracción de imágenes vacías (sin etiquetas = fondo) en el set de entrenamiento
FETCH_EXTRA_EMPTIES = "auto"  # "auto": si CCT20 trae < MIN_EMPTY_FRAC de vacías, se descargan vacías reales del CCT completo
MIN_EMPTY_FRAC      = 0.05
N_EXTRA_EMPTY       = {"train": 2500, "cis_test": 1500, "trans_test": 1500}  # vacías extra por partición (mismas ubicaciones)

# --- Aumento de datos (documentado para el informe) ---
AUG = dict(
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,   # saturación moderada: muchas fotos nocturnas son IR en escala de grises
    degrees=0.0, shear=0.0, perspective=0.0,  # la cámara está fija: no se rotan ni deforman escenas
    translate=0.1, scale=0.5,             # robustez a tamaño/posición del animal
    fliplr=0.5, flipud=0.0,               # volteo horizontal plausible; vertical no
    mosaic=1.0, close_mosaic=5,           # mosaico mezcla fondos de distintas cámaras (reduce memorizar el fondo)
    mixup=0.0,
)

# --- Ablaciones (entregables 1 y 2): mismas épocas para todas para comparar de forma justa ---
RUN_ABLATIONS = True
ABLATIONS     = ["base", "sin_vacias", "rfs"]   # base: igual al principal | sin_vacias: BG_RATIO=0 | rfs: Repeat Factor Sampling
ABL_EPOCHS    = 12
RFS_THRESHOLD = 0.10   # umbral t de Repeat Factor Sampling (Gupta et al., LVIS, CVPR 2019)

# --- Evaluación y análisis ---
CONF_EVAL      = 0.001   # umbral bajo para calcular mAP (protocolo COCO)
EVAL_MIN_CONF  = 0.01    # umbral para la evaluación propia por subgrupos
CONF_OPERATIVO = None    # umbral de operación; None = se elige maximizando F1 en validación
GRADCAM_LAYERS = [6, 8]  # capas del backbone de YOLOv8 (C2f de P4 y P5)
N_GRADCAM      = 150     # imágenes por partición para el análisis cuantitativo de Grad-CAM
N_BGSWAP       = 300     # ejemplos por dirección en la prueba de intercambio de fondo
INDEP_MINUTES  = 30      # umbral de independencia entre eventos (estándar en ecología de cámaras trampa)
REPORT_PERIOD  = "W"     # periodo del reporte de actividad: "D" día, "W" semana, "M" mes
INCLUDE_WEIGHTS_IN_ZIP = False   # los pesos quedan en Drive; actívalo si quieres best.pt dentro del zip

# --- Almacenamiento ---
IN_COLAB  = "google.colab" in sys.modules
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/P3_CCT_YOLO"
CACHE_DATASET_IN_DRIVE = True   # guarda el dataset convertido (~3-4 GB) para no repetir la descarga tras una desconexión
ULTRALYTICS_VERSION = "8.4.183"

if SMOKE_TEST:
    MODEL_WEIGHTS, IMGSZ, EPOCHS, BATCH, PATIENCE = "yolov8n.pt", 320, 2, 8, 5
    ABL_EPOCHS, N_GRADCAM, N_BGSWAP = 1, 12, 20
    FETCH_EXTRA_EMPTIES, CACHE_DATASET_IN_DRIVE, USE_DRIVE = False, False, False

random.seed(SEED)
print(f"Perfil={PROFILE} modelo={MODEL_WEIGHTS} imgsz={IMGSZ} epochs={EPOCHS} batch={BATCH} smoke={SMOKE_TEST} colab={IN_COLAB}")

In [ ]:
#@title 2. Google Drive, rutas e instalación
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = Path(DRIVE_DIR)
else:
    PERSIST = Path("./work_smoke/persist" if SMOKE_TEST else "./work/persist").resolve()

WORK     = Path("/content/work") if IN_COLAB else PERSIST.parent
RAW      = WORK / "raw"          # descargas originales
YOLO_DIR = WORK / "cct_yolo"     # dataset convertido a formato YOLO
RUNS     = PERSIST / "runs"      # entrenamientos (pesos, results.csv)
PREDS    = PERSIST / "preds"     # predicciones en caché
STATE    = PERSIST / "state"     # tablas de metadatos
OUT      = PERSIST / "outputs"   # TODO lo que va al zip
for p in [RAW, YOLO_DIR, RUNS, PREDS, STATE, OUT]:
    p.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)

import numpy as np, pandas as pd
import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2, torch, yaml
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from scipy import stats
import ultralytics
from ultralytics import YOLO
try:
    from ultralytics import settings as _ul_settings
    _ul_settings.update({"sync": False})
except Exception:
    pass

np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE  = 0 if torch.cuda.is_available() else "cpu"
HALF    = torch.cuda.is_available()
WORKERS = min(8, os.cpu_count() or 2)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 220, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

# ---------- utilidades de exportación ----------
def od(sub):
    d = OUT / sub; d.mkdir(parents=True, exist_ok=True); return d

def save_fig(fig, sub, name):
    path = od(sub) / f"{name}.png"
    fig.savefig(path, bbox_inches="tight"); plt.show() if IN_COLAB else None; plt.close(fig); return path

def save_csv(df, sub, name, index=False):
    path = od(sub) / f"{name}.csv"; df.to_csv(path, index=index); return path

RESUMEN_PATH = OUT / "resumen.json"
RESULTS = json.loads(RESUMEN_PATH.read_text()) if RESUMEN_PATH.exists() else {}
def log_result(section, key, value):
    def conv(v):
        if isinstance(v, (np.floating,)): return float(v)
        if isinstance(v, (np.integer,)):  return int(v)
        if isinstance(v, dict): return {str(k): conv(x) for k, x in v.items()}
        if isinstance(v, (list, tuple)): return [conv(x) for x in v]
        return v
    RESULTS.setdefault(section, {})[key] = conv(value)
    RESUMEN_PATH.write_text(json.dumps(RESULTS, indent=2, ensure_ascii=False))

cfg = {k: v for k, v in dict(globals()).items() if k.isupper() and isinstance(v, (int, float, str, bool, list, dict, type(None)))}
(od("00_config") / "config.json").write_text(json.dumps(cfg, indent=2, ensure_ascii=False, default=str))
env = dict(python=platform.python_version(), torch=torch.__version__, ultralytics=ultralytics.__version__,
           cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
           cpu_count=os.cpu_count(), fecha=dt.datetime.now().isoformat(timespec="seconds"))
(od("00_config") / "entorno.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2)); print("Salidas en:", OUT)

## 3. Descarga del dataset (CCT20 desde LILA BC)
Se descargan las imágenes de referencia CCT20 (≈6 GB, ya reducidas a ≤1024 px), los JSON de particiones (formato *COCO Camera Traps*) y el archivo de particiones recomendado. Si CCT20 no trae suficientes imágenes vacías, se completan con imágenes vacías reales del CCT completo **de las mismas ubicaciones** (descarga imagen por imagen desde el bucket público).

In [ ]:
#@title 3. Descarga y extracción
LILA_GCP   = "https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/"
LILA_AZURE = "https://lilawildlife.blob.core.windows.net/lila-wildlife/caltechcameratraps/"
FILES = {
    "images": "eccv_18_all_images_sm.tar.gz",
    "annotations": "eccv_18_annotations.tar.gz",
    "splits": "CaltechCameraTrapsSplits_v0.json",
    "full_meta": "labels/caltech_camera_traps.json.zip",
}
IMG_BASES = ["https://storage.googleapis.com/public-datasets-lila/caltech-unzipped/cct_images/",
             "https://lilawildlife.blob.core.windows.net/lila-wildlife/caltech-unzipped/cct_images/"]

def download(name, dest):
    import requests
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    part = dest.with_name(dest.name + ".part")
    for base in [LILA_GCP, LILA_AZURE]:
        url = base + name
        try:
            pos = part.stat().st_size if part.exists() else 0
            hdr = {"Range": f"bytes={pos}-"} if pos else {}
            with requests.get(url, stream=True, headers=hdr, timeout=60) as r:
                r.raise_for_status()
                mode = "ab" if (pos and r.status_code == 206) else "wb"
                if mode == "wb": pos = 0
                total = int(r.headers.get("content-length", 0)) + pos
                with open(part, mode) as f, tqdm(total=total, initial=pos, unit="B", unit_scale=True, desc=dest.name) as bar:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk); bar.update(len(chunk))
            part.rename(dest); return dest
        except Exception as e:
            print(f"Fallo descargando {url}: {e}")
    raise RuntimeError(f"No se pudo descargar {name}")

def extract(archive, dest, marker):
    dest = Path(dest); m = dest / marker
    if m.exists(): return dest
    print("Extrayendo", archive, "...")
    with tarfile.open(archive) as t:
        try: t.extractall(dest, filter="data")
        except TypeError: t.extractall(dest)
    m.write_text("ok"); return dest

def make_synthetic_cct(root, seed=0):
    """Dataset falso con la MISMA estructura que CCT20 (solo para SMOKE_TEST)."""
    rng = np.random.default_rng(seed)
    img_dir = root / "eccv_18_all_images_sm"; ann_dir = root / "eccv_18_annotation_files"
    img_dir.mkdir(parents=True, exist_ok=True); ann_dir.mkdir(parents=True, exist_ok=True)
    cats = [{"id": 0, "name": "empty"}, {"id": 1, "name": "opossum"}, {"id": 2, "name": "raccoon"},
            {"id": 3, "name": "coyote"}, {"id": 5, "name": "bobcat"}, {"id": 7, "name": "rabbit"}]
    colors = {1: (200, 200, 200), 2: (60, 60, 60), 3: (150, 110, 60), 5: (180, 140, 90), 7: (120, 90, 70)}
    probs = np.array([0.45, 0.25, 0.15, 0.10, 0.05])
    splits = {"train": (["38", "46", "100", "43"], 300), "cis_val": (["38", "46", "100", "43"], 60),
              "trans_val": (["61", "88"], 60), "cis_test": (["38", "46", "100", "43"], 120), "trans_test": (["33", "51", "120"], 150)}
    loc_color = {l: rng.integers(30, 220, 3) for s in splits.values() for l in s[0]}
    W0, H0 = 2048, 1494; w, h = 480, 350
    aid = 0; t0 = dt.datetime(2012, 3, 1, 6, 0, 0)
    for split, (locs, n) in splits.items():
        images, anns = [], []
        n_seq = n // 3
        for s in range(n_seq):
            loc = locs[s % len(locs)]
            seq_id = f"seq_{split}_{s}"
            t = t0 + dt.timedelta(hours=int(rng.integers(0, 24 * 40)))
            empty = rng.random() < 0.18
            cid = int(rng.choice([1, 2, 3, 5, 7], p=probs))
            night = t.hour < 6 or t.hour > 19
            for fr in range(3):
                iid = f"{split}_{s}_{fr}"
                base = np.zeros((h, w, 3), np.uint8) + loc_color[loc].astype(np.uint8)
                base = (base * np.linspace(0.6, 1.2, h)[:, None, None]).clip(0, 255).astype(np.uint8)
                base = cv2.add(base, rng.integers(0, 40, (h, w, 3), dtype=np.uint8))
                for _ in range(6):  # "vegetación"
                    c = tuple(int(x) for x in rng.integers(0, 255, 3))
                    cv2.line(base, tuple(int(x) for x in rng.integers(0, w, 2)), tuple(int(x) for x in rng.integers(0, h, 2)), c, 3)
                images.append({"id": iid, "file_name": f"{iid}.jpg", "width": W0, "height": H0, "location": loc,
                               "date_captured": (t + dt.timedelta(seconds=fr)).strftime("%Y-%m-%d %H:%M:%S"),
                               "seq_id": seq_id, "seq_num_frames": 3, "frame_num": fr})
                if empty:
                    anns.append({"id": f"a{aid}", "image_id": iid, "category_id": 0}); aid += 1
                else:
                    k = 2 if rng.random() < 0.15 else 1
                    for _ in range(k):
                        bw, bh = int(rng.integers(40, 160)), int(rng.integers(30, 120))
                        x, y = int(rng.integers(0, w - bw)), int(rng.integers(0, h - bh))
                        cv2.ellipse(base, (x + bw // 2, y + bh // 2), (bw // 2, bh // 2), 0, 0, 360, colors[cid], -1)
                        cv2.circle(base, (x + bw // 4, y + bh // 3), max(3, bh // 8), (0, 0, 0), -1)
                        sx, sy = W0 / w, H0 / h
                        anns.append({"id": f"a{aid}", "image_id": iid, "category_id": cid,
                                     "bbox": [x * sx, y * sy, bw * sx, bh * sy]}); aid += 1
                if night:
                    base = cv2.cvtColor(cv2.cvtColor(base, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)
                cv2.imwrite(str(img_dir / f"{iid}.jpg"), base)
        json.dump({"info": {}, "images": images, "annotations": anns, "categories": cats},
                  open(ann_dir / f"{split}_annotations.json", "w"))
    return img_dir, ann_dir

DATASET_READY = (YOLO_DIR / "READY").exists()
CACHE_TAR = PERSIST / "cct_yolo_cache.tar"
if not DATASET_READY and CACHE_DATASET_IN_DRIVE and CACHE_TAR.exists() and (STATE / "imgs.csv.gz").exists():
    print("Restaurando dataset convertido desde Drive...")
    with tarfile.open(CACHE_TAR) as t:
        try: t.extractall(WORK, filter="data")
        except TypeError: t.extractall(WORK)
    DATASET_READY = (YOLO_DIR / "READY").exists()

if not DATASET_READY:
    if SMOKE_TEST:
        IMG_SRC_DIR, ANN_DIR = make_synthetic_cct(RAW / "synthetic")
        SPLITS_JSON = None
    else:
        a_img = download(FILES["images"], RAW / FILES["images"])
        a_ann = download(FILES["annotations"], RAW / FILES["annotations"])
        SPLITS_JSON = download(FILES["splits"], RAW / FILES["splits"])
        extract(a_ann, RAW / "ann", ".extracted")
        extract(a_img, RAW / "img", ".extracted")
        IMG_SRC_DIR, ANN_DIR = RAW / "img", RAW / "ann"
    print("Imágenes en:", IMG_SRC_DIR, "| anotaciones en:", ANN_DIR)
else:
    print("Dataset YOLO ya preparado; se omite la descarga.")

## 4. Lectura de anotaciones y estadísticas del dataset
Cada imagen se clasifica como **animal** (todas sus anotaciones de animal tienen caja), **vacía** (solo la categoría `empty` o sin anotaciones) o **sin_caja** (tiene una especie pero no caja; se excluye de detección porque produciría falsos positivos artificiales). Se verifica que las ubicaciones *trans* no aparezcan en entrenamiento.

In [ ]:
#@title 4. Parseo de JSON COCO Camera Traps
SPLIT_NAMES = ["train", "cis_val", "trans_val", "cis_test", "trans_test"]
safe_key = lambda s: re.sub(r"[^A-Za-z0-9_.-]", "_", str(s))

def find_split_jsons(ann_dir):
    found = {}
    for p in Path(ann_dir).rglob("*.json"):
        n = p.name.lower()
        for s in ["cis_val", "trans_val", "cis_test", "trans_test"]:
            if s in n: found[s] = p
        if "train" in n and not any(s in n for s in ["val", "test"]): found["train"] = p
    missing = [s for s in SPLIT_NAMES if s not in found]
    if missing:
        raise FileNotFoundError(f"Faltan JSON de particiones {missing}. Encontrados: {[str(p) for p in Path(ann_dir).rglob('*.json')]}")
    return found

def load_cct(ann_dir, img_dir):
    imgs_l, anns_l, cats = [], [], {}
    for split, p in find_split_jsons(ann_dir).items():
        d = json.load(open(p))
        for c in d["categories"]: cats[int(c["id"])] = c["name"]
        im = pd.DataFrame(d["images"]); im["split"] = split
        an = pd.DataFrame(d["annotations"]); an["split"] = split
        imgs_l.append(im); anns_l.append(an)
    imgs = pd.concat(imgs_l, ignore_index=True); anns = pd.concat(anns_l, ignore_index=True)
    imgs["image_id"] = imgs["id"].astype(str); anns["image_id"] = anns["image_id"].astype(str)
    dtcol = "date_captured" if "date_captured" in imgs else ("datetime" if "datetime" in imgs else None)
    imgs["datetime"] = pd.to_datetime(imgs[dtcol], errors="coerce") if dtcol else pd.NaT
    imgs["location"] = imgs["location"].astype(str) if "location" in imgs else "desconocida"
    if "seq_id" not in imgs: imgs["seq_id"] = imgs["image_id"]
    imgs["seq_id"] = imgs["seq_id"].fillna(imgs["image_id"]).astype(str)
    if "frame_num" not in imgs: imgs["frame_num"] = 0
    for c in ["width", "height"]:
        if c not in imgs: imgs[c] = np.nan
    dup = imgs.duplicated("image_id").sum()
    if dup: print(f"AVISO: {dup} imágenes repetidas entre particiones; se conserva la primera."); imgs = imgs.drop_duplicates("image_id")
    # índice de archivos
    index = {}
    for p in Path(img_dir).rglob("*"):
        if p.suffix.lower() in (".jpg", ".jpeg", ".png"):
            index[p.name] = p; index.setdefault(p.stem, p)
    def resolve(r):
        fn = str(r["file_name"]) if "file_name" in r and pd.notna(r["file_name"]) else ""
        return index.get(Path(fn).name) or index.get(Path(fn).stem) or index.get(r["image_id"] + ".jpg") or index.get(r["image_id"])
    imgs["path"] = [str(x) if x is not None else None for x in (resolve(r) for _, r in imgs.iterrows())]
    imgs["source"] = "cct20"
    anns["category_id"] = anns["category_id"].astype(int)
    anns["cat_name"] = anns["category_id"].map(cats)
    if "bbox" not in anns: anns["bbox"] = None
    return imgs, anns, cats

def image_status(imgs, anns):
    a = anns.copy()
    a["is_empty_cat"] = a["cat_name"].str.lower().eq("empty")
    a["has_bbox"] = a["bbox"].apply(lambda b: isinstance(b, (list, tuple)) and len(b) == 4)
    a["n_animal_box"] = (a.has_bbox & ~a.is_empty_cat).astype(int)
    a["n_animal_nobox"] = (~a.has_bbox & ~a.is_empty_cat).astype(int)
    g = a.groupby("image_id")[["n_animal_box", "n_animal_nobox"]].sum()
    imgs = imgs.drop(columns=[c for c in ["n_animal_box", "n_animal_nobox", "status"] if c in imgs]).merge(g, left_on="image_id", right_index=True, how="left")
    imgs[["n_animal_box", "n_animal_nobox"]] = imgs[["n_animal_box", "n_animal_nobox"]].fillna(0).astype(int)
    imgs["status"] = np.where((imgs.n_animal_box > 0) & (imgs.n_animal_nobox == 0), "animal",
                     np.where((imgs.n_animal_box == 0) & (imgs.n_animal_nobox == 0), "vacia", "sin_caja"))
    return imgs

if not DATASET_READY:
    imgs, anns, CATS = load_cct(ANN_DIR, IMG_SRC_DIR)
    imgs = image_status(imgs, anns)
    n_missing = imgs["path"].isna().sum()
    print(f"Imágenes: {len(imgs)} | sin archivo: {n_missing} | anotaciones: {len(anns)}")
    imgs = imgs[imgs["path"].notna()].copy()
    print(imgs.groupby(["split", "status"]).size().unstack(fill_value=0))

In [ ]:
#@title 4b. Vacías extra del CCT completo (solo si CCT20 casi no trae vacías)
def fetch_extra_empties(imgs):
    import requests
    zp = download(FILES["full_meta"], RAW / "caltech_camera_traps.json.zip")
    with zipfile.ZipFile(zp) as z:
        name = [n for n in z.namelist() if n.endswith(".json")][0]
        d = json.load(z.open(name))
    fcats = {int(c["id"]): c["name"].lower() for c in d["categories"]}
    fim = pd.DataFrame(d["images"]); fan = pd.DataFrame(d["annotations"])
    fim["image_id"] = fim["id"].astype(str); fan["image_id"] = fan["image_id"].astype(str)
    fan["is_empty"] = fan["category_id"].astype(int).map(fcats).eq("empty")
    only_empty = fan.groupby("image_id")["is_empty"].all()
    fim = fim[fim.image_id.map(only_empty).fillna(False).astype(bool)].copy()
    fim["location"] = fim["location"].astype(str)
    fim["seq_id"] = fim.get("seq_id", fim["image_id"]).astype(str)
    fim = fim[~fim.image_id.isin(set(imgs.image_id)) & ~fim.seq_id.isin(set(imgs.seq_id))]
    seen = set(imgs.loc[imgs.split == "train", "location"]); unseen = set(imgs.loc[imgs.split == "trans_test", "location"])
    print(f"Vacías candidatas en ubicaciones vistas: {fim.location.isin(seen).sum()} | no vistas (trans_test): {fim.location.isin(unseen).sum()}")
    if fim.location.isin(seen).sum() == 0:
        print("AVISO: las ubicaciones no coinciden entre CCT20 y CCT completo; no se agregan vacías extra."); return imgs
    rng = np.random.default_rng(SEED); chosen = []
    def take_by_seq(pool, n, used):
        seqs = [s for s in pool.seq_id.unique() if s not in used]; rng.shuffle(seqs)
        sel, cnt = [], 0
        by = pool.groupby("seq_id").groups
        for s in seqs:
            if cnt >= n: break
            sel.append(s); cnt += len(by[s])
        used.update(sel); return pool[pool.seq_id.isin(sel)].head(n)
    used = set()
    pool_seen = fim[fim.location.isin(seen)]; pool_unseen = fim[fim.location.isin(unseen)]
    for split, pool in [("train", pool_seen), ("cis_test", pool_seen), ("trans_test", pool_unseen)]:
        sub = take_by_seq(pool, N_EXTRA_EMPTY[split], used).copy(); sub["split"] = split; chosen.append(sub)
    ext = pd.concat(chosen, ignore_index=True)
    out_dir = RAW / "extra_empty"; out_dir.mkdir(exist_ok=True)
    sess = requests.Session()
    def fetch(r):
        dst = out_dir / f"{safe_key(r.image_id)}.jpg"
        if dst.exists(): return str(dst)
        fn = str(r.file_name).replace("cct_images/", "")
        for base in IMG_BASES:
            try:
                resp = sess.get(base + fn, timeout=30)
                if resp.status_code == 200:
                    arr = cv2.imdecode(np.frombuffer(resp.content, np.uint8), cv2.IMREAD_COLOR)
                    s = 1024 / max(arr.shape[:2])
                    if s < 1: arr = cv2.resize(arr, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
                    cv2.imwrite(str(dst), arr, [cv2.IMWRITE_JPEG_QUALITY, 92]); return str(dst)
            except Exception:
                pass
        return None
    with ThreadPoolExecutor(32) as ex:
        paths = list(tqdm(ex.map(fetch, [r for r in ext.itertuples()]), total=len(ext), desc="vacías extra"))
    ext["path"] = paths; ext = ext[ext.path.notna()]
    dtcol = "date_captured" if "date_captured" in ext else "datetime"
    ext = pd.DataFrame({"image_id": ext.image_id, "file_name": ext.file_name, "width": np.nan, "height": np.nan,
                        "location": ext.location, "datetime": pd.to_datetime(ext.get(dtcol), errors="coerce"),
                        "seq_id": ext.seq_id, "frame_num": ext.get("frame_num", 0), "split": ext.split, "path": ext.path,
                        "source": "cct_full_empty", "n_animal_box": 0, "n_animal_nobox": 0, "status": "vacia"})
    print("Vacías extra descargadas por partición:", ext.split.value_counts().to_dict())
    return pd.concat([imgs, ext], ignore_index=True)

if not DATASET_READY:
    empty_frac_train = (imgs.loc[imgs.split == "train", "status"] == "vacia").mean()
    print(f"Fracción de vacías en train (CCT20): {empty_frac_train:.3f}")
    do_fetch = FETCH_EXTRA_EMPTIES is True or (FETCH_EXTRA_EMPTIES == "auto" and empty_frac_train < MIN_EMPTY_FRAC)
    if do_fetch and not SMOKE_TEST:
        imgs = fetch_extra_empties(imgs)
    log_result("dataset", "fraccion_vacias_train_cct20", float(empty_frac_train))
    log_result("dataset", "vacias_extra_descargadas", bool(do_fetch and not SMOKE_TEST))

## 5. Conversión a formato YOLO
* Las imágenes se re-muestrean a lado mayor = `IMGSZ` (acelera mucho la carga en Colab, que solo tiene 2 CPU; YOLO las redimensiona a `IMGSZ` de todos modos).
* Las cajas COCO `[x, y, w, h]` (en píxeles de la imagen original) se normalizan con el tamaño declarado en el JSON, de modo que son válidas aunque las imágenes `sm` estén reducidas.
* Las **imágenes vacías** se escriben con un archivo de etiquetas vacío: YOLO las trata como **fondo** (solo contribuyen a la pérdida de objetividad/clasificación como negativos).
* En el mismo recorrido se calculan, por imagen, si es nocturna (IR) y, por caja, la densidad de bordes (insumo del proxy de oclusión de la sección 12).

In [ ]:
#@title 5. Conversión, listas de entrenamiento y YAML
def build_yolo(imgs, anns, cats):
    animal_cats = sorted(int(c) for c in anns.loc[anns.bbox.apply(lambda b: isinstance(b, (list, tuple))), "category_id"].unique()
                         if str(cats[int(c)]).lower() != "empty")
    cat2cls = {c: i for i, c in enumerate(animal_cats)}
    names = {i: cats[c] for c, i in cat2cls.items()}
    use = imgs[imgs.status.isin(["animal", "vacia"])].copy()
    use["key"] = use.image_id.map(safe_key)
    box_anns = anns[anns.bbox.apply(lambda b: isinstance(b, (list, tuple)) and len(b) == 4) & anns.category_id.isin(cat2cls)]
    by_img = {k: g for k, g in box_anns.groupby("image_id")}
    rows_meta, rows_gt = [], []
    def process(r):
        dst = YOLO_DIR / "images" / r.split / f"{r.key}.jpg"
        lab = YOLO_DIR / "labels" / r.split / f"{r.key}.txt"
        im = cv2.imread(r.path, cv2.IMREAD_COLOR)
        if im is None: return None
        H0, W0 = im.shape[:2]
        s = IMGSZ / max(H0, W0)
        if s < 1: im = cv2.resize(im, (round(W0 * s), round(H0 * s)), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(dst), im, [cv2.IMWRITE_JPEG_QUALITY, 93])
        h, w = im.shape[:2]
        small = cv2.resize(im, (64, 64)).astype(np.int16)
        chan_diff = float(np.abs(small[..., 0] - small[..., 1]).mean() + np.abs(small[..., 1] - small[..., 2]).mean())
        gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY); edges = cv2.Canny(gray, 80, 160) > 0
        jw = r.width if pd.notna(r.width) and r.width > 0 else W0
        jh = r.height if pd.notna(r.height) and r.height > 0 else H0
        gts, lines, oob = [], [], 0
        for a in (by_img[r.image_id].itertuples() if r.image_id in by_img else []):
            x, y, bw, bh = a.bbox
            x1, y1, x2, y2 = x / jw, y / jh, (x + bw) / jw, (y + bh) / jh
            if x2 > 1.02 or y2 > 1.02: oob += 1
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(1, x2), min(1, y2)
            if (x2 - x1) * w < 2 or (y2 - y1) * h < 2: continue
            c = cat2cls[int(a.category_id)]
            lines.append(f"{c} {(x1 + x2) / 2:.6f} {(y1 + y2) / 2:.6f} {x2 - x1:.6f} {y2 - y1:.6f}")
            px = edges[int(y1 * h):max(int(y1 * h) + 1, int(y2 * h)), int(x1 * w):max(int(x1 * w) + 1, int(x2 * w))]
            gts.append(dict(key=r.key, split=r.split, cls=c, x1=x1, y1=y1, x2=x2, y2=y2, edge_density=float(px.mean())))
        lab.write_text("\n".join(lines))
        return dict(key=r.key, img_w=w, img_h=h, is_night=chan_diff < 4.0, oob=oob, yolo_path=str(dst)), gts
    for s in use.split.unique():
        (YOLO_DIR / "images" / s).mkdir(parents=True, exist_ok=True); (YOLO_DIR / "labels" / s).mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max(4, WORKERS * 2)) as ex:
        for res in tqdm(ex.map(process, use.itertuples()), total=len(use), desc="convirtiendo"):
            if res is None: continue
            rows_meta.append(res[0]); rows_gt.extend(res[1])
    meta = pd.DataFrame(rows_meta)
    use = use.merge(meta, on="key", how="inner")
    gt = pd.DataFrame(rows_gt, columns=["key", "split", "cls", "x1", "y1", "x2", "y2", "edge_density"])
    gt = gt.merge(use[["key", "location"]], on="key", how="left")
    print(f"Cajas fuera de rango (>2% del borde): {int(use.oob.sum())} (si es alto, las cajas no están en la escala esperada)")
    return use, gt, names

def write_list(paths, name):
    p = YOLO_DIR / f"{name}.txt"; p.write_text("\n".join(map(str, paths))); return p

def make_rfs_list(train_df, gt, t=RFS_THRESHOLD, seed=SEED):
    """Repeat Factor Sampling: r_c = max(1, sqrt(t/f_c)); r_img = max_c r_c; repeticiones con redondeo estocástico."""
    rng = np.random.default_rng(seed)
    g = gt[gt.split == "train"]
    n_img = train_df.key.nunique()
    f_c = g.groupby("cls").key.nunique() / n_img
    r_c = np.maximum(1.0, np.sqrt(t / f_c))
    r_img = g.assign(r=g.cls.map(r_c)).groupby("key").r.max()
    out = []
    for r in train_df.itertuples():
        rep = r_img.get(r.key, 1.0)
        k = int(np.floor(rep)) + int(rng.random() < rep - np.floor(rep))
        out.append(r.yolo_path)
        for j in range(1, k):  # copias con nombre distinto (symlink + etiqueta)
            src = Path(r.yolo_path); dst = src.with_name(f"{src.stem}__rep{j}.jpg")
            lsrc = Path(str(src).replace("/images/", "/labels/")).with_suffix(".txt")
            ldst = Path(str(dst).replace("/images/", "/labels/")).with_suffix(".txt")
            if not dst.exists(): os.symlink(src, dst)
            if not ldst.exists(): shutil.copy(lsrc, ldst)
            out.append(str(dst))
    return out, r_c

if not DATASET_READY:
    data, gt, NAMES = build_yolo(imgs, anns, CATS)
    rng = np.random.default_rng(SEED)
    tr = data[data.split == "train"]
    tr_animal, tr_empty = tr[tr.status == "animal"], tr[tr.status == "vacia"]
    n_bg = int(round(BG_RATIO / (1 - BG_RATIO) * len(tr_animal)))
    if n_bg > len(tr_empty): print(f"AVISO: solo hay {len(tr_empty)} vacías en train (se pedían {n_bg}).")
    bg = tr_empty.sample(min(n_bg, len(tr_empty)), random_state=SEED)
    data["in_train"] = data.key.isin(set(tr_animal.key) | set(bg.key))
    write_list(tr_animal.yolo_path.tolist() + bg.yolo_path.tolist(), "train_base")
    write_list(tr_animal.yolo_path.tolist(), "train_sin_vacias")
    rfs_paths, r_c = make_rfs_list(data[data.in_train], gt)
    write_list(rfs_paths, "train_rfs")
    write_list(data[data.split.isin(["cis_val", "trans_val"])].yolo_path.tolist(), "val")
    for s in ["cis_val", "trans_val", "cis_test", "trans_test"]:
        write_list(data[data.split == s].yolo_path.tolist(), s)
    data.to_csv(STATE / "imgs.csv.gz", index=False); gt.to_csv(STATE / "gt.csv.gz", index=False)
    (STATE / "names.json").write_text(json.dumps(NAMES))
    pd.Series(r_c, name="repeat_factor").rename_axis("cls").to_csv(STATE / "rfs_factors.csv")
    (YOLO_DIR / "READY").write_text("ok")
    if CACHE_DATASET_IN_DRIVE:
        print("Guardando caché del dataset convertido en Drive (puede tardar unos minutos)...")
        with tarfile.open(CACHE_TAR, "w") as t: t.add(YOLO_DIR, arcname=YOLO_DIR.name)

data = pd.read_csv(STATE / "imgs.csv.gz", parse_dates=["datetime"], dtype={"location": str, "seq_id": str, "image_id": str, "key": str})
gt = pd.read_csv(STATE / "gt.csv.gz", dtype={"location": str, "key": str})
NAMES = {int(k): v for k, v in json.loads((STATE / "names.json").read_text()).items()}
NC = len(NAMES)
key2split = dict(zip(data.key, data.split))

def make_yaml(train_txt, val_txt, name):
    p = YOLO_DIR / f"{name}.yaml"
    p.write_text(yaml.safe_dump({"path": str(YOLO_DIR), "train": str(train_txt), "val": str(val_txt), "names": NAMES}, allow_unicode=True))
    return p
for s in ["cis_val", "trans_val", "cis_test", "trans_test"]:
    make_yaml(YOLO_DIR / "train_base.txt", YOLO_DIR / f"{s}.txt", f"eval_{s}")
print("Clases:", NAMES)

In [ ]:
#@title 5b. Estadísticas del dataset (tablas y figuras para el informe)
SUB = "01_dataset"
split_order = SPLIT_NAMES
st = data.groupby(["split", "status"]).size().unstack(fill_value=0).reindex(split_order).fillna(0).astype(int)
st["total"] = st.sum(1); st["pct_vacias"] = (100 * st.get("vacia", 0) / st.total).round(1)
st["n_ubicaciones"] = data.groupby("split").location.nunique().reindex(split_order)
st["pct_nocturnas"] = (100 * data.groupby("split").is_night.mean()).round(1).reindex(split_order)
save_csv(st.reset_index(), SUB, "imagenes_por_particion"); print(st)

gt["cls_name"] = gt.cls.map(NAMES)
cls_counts = gt.pivot_table(index="cls_name", columns="split", values="key", aggfunc="count", fill_value=0).reindex(columns=split_order, fill_value=0)
cls_counts = cls_counts.sort_values("train", ascending=False)
tr_keys = set(data.loc[data.in_train, "key"])
cls_counts["imgs_train"] = gt[gt.key.isin(tr_keys)].groupby("cls_name").key.nunique().reindex(cls_counts.index).fillna(0).astype(int)
cls_counts["prevalencia_train_%"] = (100 * cls_counts.train / cls_counts.train.sum()).round(2)
save_csv(cls_counts.reset_index(), SUB, "cajas_por_especie_y_particion")

locs = data.groupby("split").location.apply(lambda s: sorted(set(s)))
loc_tab = pd.DataFrame({"split": locs.index, "ubicaciones": locs.apply(lambda l: ", ".join(l)), "n": locs.apply(len)})
save_csv(loc_tab, SUB, "ubicaciones_por_particion")
overlap = {f"{a}∩{b}": sorted(set(locs.get(a, [])) & set(locs.get(b, []))) for a in split_order for b in split_order if a < b}
log_result("dataset", "solapamiento_ubicaciones", {k: len(v) for k, v in overlap.items()})
leak = set(locs.get("train", [])) & set(locs.get("trans_test", []))
print("Ubicaciones trans_test que aparecen en train:", leak or "ninguna ✔")

fig, ax = plt.subplots(figsize=(7, 0.32 * len(cls_counts) + 1.2))
yy = np.arange(len(cls_counts)); hgt = 0.18
for i, s in enumerate(["train", "cis_test", "trans_test"]):
    ax.barh(yy + (i - 1) * hgt, cls_counts[s].clip(lower=0.8), height=hgt, label=s)
ax.set_yticks(yy); ax.set_yticklabels(cls_counts.index); ax.invert_yaxis(); ax.set_xscale("log")
ax.set_xlabel("Número de cajas (escala log)"); ax.set_title("Distribución de especies por partición"); ax.legend()
save_fig(fig, SUB, "distribucion_especies")

gt["rel_area"] = (gt.x2 - gt.x1) * (gt.y2 - gt.y1)
fig, axs = plt.subplots(1, 2, figsize=(9, 3))
axs[0].hist(np.log10(gt.rel_area.clip(1e-5)), bins=50); axs[0].set_xlabel("log10(área relativa de la caja)"); axs[0].set_ylabel("cajas")
bpi = gt.groupby("key").size(); axs[1].hist(bpi, bins=range(1, bpi.max() + 2)); axs[1].set_xlabel("cajas por imagen con animal")
save_fig(fig, SUB, "tamano_cajas_y_cajas_por_imagen")

# muestras con cajas GT (verificación visual de que la escala de las cajas es correcta)
def draw_boxes(img, boxes, color, labels=None, thick=2):
    h, w = img.shape[:2]
    for i, b in enumerate(boxes):
        p1, p2 = (int(b[0] * w), int(b[1] * h)), (int(b[2] * w), int(b[3] * h))
        cv2.rectangle(img, p1, p2, color, thick)
        if labels is not None:
            cv2.putText(img, labels[i], (p1[0], max(12, p1[1] - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return img
sample = data[(data.status == "animal")].sample(min(12, (data.status == "animal").sum()), random_state=SEED)
fig, axs = plt.subplots(3, 4, figsize=(12, 7))
for ax, r in zip(axs.flat, sample.itertuples()):
    im = cv2.imread(r.yolo_path); g = gt[gt.key == r.key]
    draw_boxes(im, g[["x1", "y1", "x2", "y2"]].values, (0, 255, 0), g.cls_name.tolist())
    ax.imshow(im[..., ::-1]); ax.set_title(f"{r.split} · loc {r.location}", fontsize=8); ax.axis("off")
save_fig(fig, SUB, "muestras_con_cajas_gt")
log_result("dataset", "n_imagenes", st.total.to_dict())
log_result("dataset", "n_clases", NC); log_result("dataset", "clases", list(NAMES.values()))
log_result("dataset", "fuga_ubicaciones_train_trans_test", len(leak))

## 6–7. Entrenamiento de YOLOv8 (modelo principal)
Decisiones documentadas (todas quedan en `00_config/config.json` y en `args.yaml` de cada corrida):
* **Backbone**: YOLOv8 preentrenado en COCO (transfer learning); toda la red se ajusta.
* **Resolución**: `IMGSZ` (640 en el perfil estándar), el doble de lo que suelen usar los clasificadores de fauna, para no perder animales pequeños/lejanos.
* **Aumento**: mosaico (combina fondos de distintas cámaras y obliga a separar animal de fondo), HSV moderado, traslación/escala, volteo horizontal; sin rotaciones (cámara fija). Mosaico se apaga en las últimas 5 épocas.
* **Vacías como fondo**: `BG_RATIO` de las imágenes de entrenamiento son vacías con etiqueta vacía → ejemplos negativos explícitos que penalizan falsos positivos por viento, vegetación o luz.
* **Selección de modelo**: mejor mAP50-95 sobre `cis_val ∪ trans_val`; *early stopping* con paciencia `PATIENCE`.

In [ ]:
#@title 7. Entrenamiento (reanudable)
def train_run(name, train_list, epochs):
    run_dir = RUNS / name; done = run_dir / "DONE"; last = run_dir / "weights" / "last.pt"
    if done.exists():
        print(f"[{name}] ya terminado ✔"); return run_dir
    data_yaml = make_yaml(train_list, YOLO_DIR / "val.txt", f"train_{name}")
    t0 = time.time()
    if last.exists():
        print(f"[{name}] reanudando desde {last}")
        try:
            YOLO(str(last)).train(resume=True)
        except Exception as e:
            print(f"[{name}] No se pudo reanudar ({e}); se marca como terminado con los pesos existentes.")
    else:
        model = YOLO(MODEL_WEIGHTS)
        model.train(data=str(data_yaml), epochs=epochs, imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS,
                    project=str(RUNS), name=name, exist_ok=True, seed=SEED, deterministic=True, patience=PATIENCE,
                    optimizer="auto", cos_lr=False, amp=HALF, plots=True, verbose=False, **AUG)
    done.write_text(json.dumps({"segundos": time.time() - t0, "fin": dt.datetime.now().isoformat()}))
    return run_dir

MAIN_RUN = train_run("principal", YOLO_DIR / "train_base.txt", EPOCHS)
BEST = MAIN_RUN / "weights" / "best.pt"
print("Mejor checkpoint:", BEST)

In [ ]:
#@title 7b. Curvas de entrenamiento
SUB = "02_entrenamiento"
res = pd.read_csv(MAIN_RUN / "results.csv"); res.columns = [c.strip() for c in res.columns]
save_csv(res, SUB, "results_principal")
for f in ["args.yaml", "results.png", "labels.jpg", "PR_curve.png", "BoxPR_curve.png", "BoxF1_curve.png", "confusion_matrix_normalized.png"]:
    if (MAIN_RUN / f).exists(): shutil.copy(MAIN_RUN / f, od(SUB) / f"principal_{f}")
fig, axs = plt.subplots(1, 3, figsize=(12, 3))
for c in ["train/box_loss", "train/cls_loss", "train/dfl_loss"]:
    if c in res: axs[0].plot(res.epoch, res[c], label=c.split("/")[1])
for c in ["val/box_loss", "val/cls_loss", "val/dfl_loss"]:
    if c in res: axs[1].plot(res.epoch, res[c], label=c.split("/")[1])
for c in ["metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/precision(B)", "metrics/recall(B)"]:
    if c in res: axs[2].plot(res.epoch, res[c], label=c.split("/")[1].replace("(B)", ""))
for ax, t in zip(axs, ["Pérdidas de entrenamiento", "Pérdidas de validación", "Métricas de validación"]):
    ax.set_title(t); ax.set_xlabel("época"); ax.legend(fontsize=7)
save_fig(fig, SUB, "curvas_entrenamiento")
best_ep = res.loc[res["metrics/mAP50-95(B)"].idxmax()]
log_result("entrenamiento", "epocas_entrenadas", int(res.epoch.max()))
log_result("entrenamiento", "mejor_epoca", int(best_ep.epoch))
log_result("entrenamiento", "val_mAP50_mejor", float(best_ep["metrics/mAP50(B)"]))
log_result("entrenamiento", "val_mAP50_95_mejor", float(best_ep["metrics/mAP50-95(B)"]))
log_result("entrenamiento", "tiempo_horas", json.loads((MAIN_RUN / "DONE").read_text()).get("segundos", 0) / 3600)
log_result("entrenamiento", "n_imagenes_train", int(data.in_train.sum()))
log_result("entrenamiento", "n_vacias_train", int((data.in_train & (data.status == "vacia")).sum()))

## 8. Evaluación cuantitativa: mAP@0.5 y mAP@0.5:0.95 por especie
Se usa el validador oficial de Ultralytics sobre cada partición de prueba (umbral de confianza 0.001, NMS IoU 0.7) y se guardan las predicciones para los análisis siguientes.

In [ ]:
#@title 8. Validación oficial por partición y predicciones en caché
SUB = "03_evaluacion"
model = YOLO(str(BEST))

def ul_val(weights_model, split, run_name="principal"):
    m = weights_model.val(data=str(YOLO_DIR / f"eval_{split}.yaml"), split="val", imgsz=IMGSZ, batch=BATCH, conf=CONF_EVAL,
                          iou=0.7, device=DEVICE, plots=True, project=str(OUT / "_val_runs"), name=f"{run_name}_{split}",
                          exist_ok=True, verbose=False)
    rows = []
    for i, c in enumerate(m.box.ap_class_index):
        rows.append(dict(split=split, cls=int(c), especie=NAMES[int(c)], AP50=float(m.box.ap50[i]), AP50_95=float(m.box.ap[i]),
                         P=float(m.box.p[i]), R=float(m.box.r[i])))
    summ = dict(split=split, mAP50=float(m.box.map50), mAP50_95=float(m.box.map), P=float(m.box.mp), R=float(m.box.mr),
                ms_inferencia=float(m.speed.get("inference", np.nan)), ms_total=float(sum(m.speed.values())))
    try:
        cm = m.confusion_matrix.matrix
        lbl = [NAMES[i] for i in range(NC)] + ["fondo"]
        summ["_cm"] = pd.DataFrame(cm, index=[f"pred_{l}" for l in lbl], columns=[f"gt_{l}" for l in lbl])
    except Exception:
        summ["_cm"] = None
    return summ, pd.DataFrame(rows)

EVAL_SPLITS = ["cis_val", "trans_val", "cis_test", "trans_test"]
summ_rows, cls_rows = [], []
for s in EVAL_SPLITS:
    summ, per_cls = ul_val(model, s)
    if summ["_cm"] is not None: save_csv(summ["_cm"], SUB, f"matriz_confusion_{s}", index=True)
    vd = OUT / "_val_runs" / f"principal_{s}"
    for f in ["confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxF1_curve.png"]:
        if (vd / f).exists(): shutil.copy(vd / f, od(SUB) / f"{s}_{f}")
    summ.pop("_cm"); summ_rows.append(summ); cls_rows.append(per_cls)
summary_df = pd.DataFrame(summ_rows); per_class_df = pd.concat(cls_rows, ignore_index=True)
gt_counts = gt.groupby(["split", "cls"]).size().rename("n_gt_split").reset_index()
train_counts = gt[gt.key.isin(tr_keys)].groupby("cls").size().rename("n_cajas_train")
per_class_df = per_class_df.merge(gt_counts, on=["split", "cls"], how="left")
per_class_df["n_cajas_train"] = per_class_df.cls.map(train_counts).fillna(0).astype(int)
save_csv(summary_df, SUB, "resumen_por_particion"); save_csv(per_class_df, SUB, "metricas_por_especie")
print(summary_df.round(4).to_string(index=False))
for r in summary_df.itertuples():
    log_result("evaluacion", f"{r.split}_mAP50", r.mAP50); log_result("evaluacion", f"{r.split}_mAP50_95", r.mAP50_95)
    log_result("evaluacion", f"{r.split}_P", r.P); log_result("evaluacion", f"{r.split}_R", r.R)

def predict_df(m, keys_paths, conf=CONF_EVAL, bs=32, max_det=100):
    rows = []
    for i in tqdm(range(0, len(keys_paths), bs), desc="predicción"):
        chunk = keys_paths[i:i + bs]
        res = m.predict([p for _, p in chunk], imgsz=IMGSZ, conf=conf, iou=0.7, max_det=max_det, device=DEVICE,
                        verbose=False, batch=len(chunk))
        for (k, _), r in zip(chunk, res):
            b = r.boxes
            if b is None or len(b) == 0: continue
            xy = b.xyxyn.cpu().numpy(); cf = b.conf.cpu().numpy(); cl = b.cls.cpu().numpy().astype(int)
            rows.append(pd.DataFrame({"key": k, "cls": cl, "conf": cf, "x1": xy[:, 0], "y1": xy[:, 1], "x2": xy[:, 2], "y2": xy[:, 3]}))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["key", "cls", "conf", "x1", "y1", "x2", "y2"])

def cached_preds(m, run_name, split):
    f = PREDS / f"{run_name}_{split}.csv.gz"
    if f.exists(): return pd.read_csv(f, dtype={"key": str})
    d = data[data.split == split]
    p = predict_df(m, list(zip(d.key, d.yolo_path))); p.to_csv(f, index=False); return p

preds = {s: cached_preds(model, "principal", s) for s in EVAL_SPLITS}
for s in ["cis_test", "trans_test"]:
    save_csv(preds[s][preds[s].conf >= 0.25].assign(especie=lambda d: d.cls.map(NAMES)), SUB, f"predicciones_conf025_{s}")
print({s: len(p) for s, p in preds.items()})

In [ ]:
#@title 8b. Evaluador propio (COCO 101 puntos con regiones ignoradas) — se usa para subgrupos
IOU_THRS = np.round(np.linspace(0.5, 0.95, 10), 2)

def iou_matrix(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-12)

def _match(ious, gt_ign, thr):
    D, G = ious.shape; used = np.zeros(G, bool); state = np.zeros(D, np.int8); mgt = np.full(D, -1)
    order = np.argsort(gt_ign, kind="stable")
    for d in range(D):
        best, best_iou = -1, min(thr, 1 - 1e-10)
        for g in order:
            if used[g]: continue
            if best > -1 and not gt_ign[best] and gt_ign[g]: break
            if ious[d, g] < best_iou: continue
            best, best_iou = g, ious[d, g]
        if best == -1: continue
        used[best] = True; state[d] = 2 if gt_ign[best] else 1; mgt[d] = best
    return state, mgt

def _ap101(tp, fp, n_pos):
    if n_pos == 0: return np.nan
    tpc, fpc = np.cumsum(tp), np.cumsum(fp)
    rec = tpc / n_pos; prec = tpc / np.maximum(tpc + fpc, 1e-12)
    prec = np.maximum.accumulate(prec[::-1])[::-1] if len(prec) else prec
    q = np.zeros(101)
    idx = np.searchsorted(rec, np.linspace(0, 1, 101), side="left")
    for i, j in enumerate(idx):
        if j < len(prec): q[i] = prec[j]
    return q.mean()

def evaluate_det(gt_df, pred_df, gt_ignore=None, det_ignore=None, min_gt=1, conf_recall=0.25, classes=None, min_conf=EVAL_MIN_CONF):
    """AP COCO por clase. gt_ignore: bool por GT (fuera del subgrupo). det_ignore: bool por detección no emparejada a ignorar."""
    g = gt_df.copy(); g["_ign"] = False if gt_ignore is None else np.asarray(gt_ignore, bool)
    p = pred_df[pred_df.conf >= min_conf].copy()
    p["_dign"] = False if det_ignore is None else np.asarray(det_ignore, bool)[pred_df.conf.values >= min_conf]
    classes = sorted(g.cls.unique()) if classes is None else classes
    out = []
    for c in classes:
        gc = g[g.cls == c]; pc = p[p.cls == c].sort_values("conf", ascending=False, kind="stable").reset_index(drop=True)
        n_pos = int((~gc._ign).sum())
        if n_pos < min_gt: continue
        T = len(IOU_THRS); states = np.zeros((T, len(pc)), np.int8)
        gt_groups = {k: v for k, v in gc.groupby("key")}
        matched_conf = []
        for k, dg in pc.groupby("key", sort=False):
            idx = dg.index.values
            if k not in gt_groups:
                states[:, idx] = np.where(dg._dign.values, 2, 0); continue
            gg = gt_groups[k]; ious = iou_matrix(dg[["x1", "y1", "x2", "y2"]].values, gg[["x1", "y1", "x2", "y2"]].values)
            gign = gg._ign.values
            for ti, t in enumerate(IOU_THRS):
                stt, mgt = _match(ious, gign, t)
                stt = np.where((stt == 0) & dg._dign.values, 2, stt); states[ti, idx] = stt
                if ti == 0:
                    for d in np.where(stt == 1)[0]: matched_conf.append(dg.conf.values[d])
        aps = []
        for ti in range(T):
            v = states[ti] != 2
            aps.append(_ap101((states[ti][v] == 1).astype(float), (states[ti][v] == 0).astype(float), n_pos))
        out.append(dict(cls=int(c), especie=NAMES[int(c)], n_gt=n_pos, AP50=aps[0], AP50_95=float(np.nanmean(aps)),
                        recall_conf=float((np.array(matched_conf) >= conf_recall).sum() / n_pos)))
    df = pd.DataFrame(out)
    summ = dict(mAP50=df.AP50.mean() if len(df) else np.nan, mAP50_95=df.AP50_95.mean() if len(df) else np.nan,
                recall=(df.recall_conf * df.n_gt).sum() / df.n_gt.sum() if len(df) else np.nan, n_gt=int(df.n_gt.sum()) if len(df) else 0,
                n_clases=len(df))
    return summ, df

def gt_of(split): return gt[gt.split == split].reset_index(drop=True)

# Validación cruzada del evaluador propio contra Ultralytics
chk = []
for s in ["cis_test", "trans_test"]:
    sm, _ = evaluate_det(gt_of(s), preds[s], min_conf=CONF_EVAL)
    u = summary_df.set_index("split").loc[s]
    chk.append(dict(split=s, mAP50_ultralytics=u.mAP50, mAP50_propio=sm["mAP50"], mAP50_95_ultralytics=u.mAP50_95, mAP50_95_propio=sm["mAP50_95"]))
chk = pd.DataFrame(chk); save_csv(chk, SUB, "verificacion_evaluador_propio"); print(chk.round(4).to_string(index=False))

# Umbral operativo: máximo F1 (IoU 0.5, por clase) en validación
def f1_curve(gt_df, pred_df, thrs=np.round(np.arange(0.05, 0.96, 0.05), 2)):
    p = pred_df[pred_df.conf >= 0.05].sort_values("conf", ascending=False, kind="stable").reset_index(drop=True)
    tp_conf, fp_conf, n_gt = [], [], len(gt_df)
    gtg = {k: v for k, v in gt_df.groupby(["key", "cls"])}
    for (k, c), dg in p.groupby(["key", "cls"], sort=False):
        if (k, c) not in gtg: fp_conf += list(dg.conf); continue
        gg = gtg[(k, c)]; st_, _ = _match(iou_matrix(dg[["x1", "y1", "x2", "y2"]].values, gg[["x1", "y1", "x2", "y2"]].values), np.zeros(len(gg), bool), 0.5)
        tp_conf += list(dg.conf.values[st_ == 1]); fp_conf += list(dg.conf.values[st_ == 0])
    tp_conf, fp_conf = np.array(tp_conf), np.array(fp_conf); rows = []
    for t in thrs:
        tp = (tp_conf >= t).sum(); fp = (fp_conf >= t).sum(); fn = n_gt - tp
        P_ = tp / max(tp + fp, 1); R_ = tp / max(n_gt, 1); rows.append(dict(conf=t, P=P_, R=R_, F1=2 * P_ * R_ / max(P_ + R_, 1e-12)))
    return pd.DataFrame(rows)

val_gt = gt[gt.split.isin(["cis_val", "trans_val"])].reset_index(drop=True)
val_pred = pd.concat([preds["cis_val"], preds["trans_val"]], ignore_index=True)
f1v = f1_curve(val_gt, val_pred); save_csv(f1v, SUB, "curva_f1_validacion")
CONF_OP = float(f1v.loc[f1v.F1.idxmax(), "conf"]) if CONF_OPERATIVO is None else CONF_OPERATIVO
log_result("evaluacion", "umbral_operativo_conf", CONF_OP); log_result("evaluacion", "F1_val_en_umbral", float(f1v.F1.max()))
print("Umbral operativo (máx F1 en validación):", CONF_OP)
for s in ["cis_test", "trans_test"]:
    fc = f1_curve(gt_of(s), preds[s]); row = fc.iloc[(fc.conf - CONF_OP).abs().argmin()]
    log_result("evaluacion", f"{s}_P_R_F1_en_umbral", dict(P=row.P, R=row.R, F1=row.F1))

## 9. Efecto del desbalance entre especies
Se relaciona el AP de cada especie con su número de cajas de entrenamiento. Las especies se agrupan en **cabeza / media / cola** por terciles de frecuencia.

In [ ]:
#@title 9. Desbalance de clases
SUB = "03_evaluacion"
pc = per_class_df[per_class_df.split.isin(["cis_test", "trans_test"])].copy()
tc = train_counts.reindex(range(NC)).fillna(0)
q1, q2 = np.quantile(tc[tc > 0], [1 / 3, 2 / 3]) if (tc > 0).sum() >= 3 else (tc.max(), tc.max())
grp = lambda n: "cola" if n <= q1 else ("media" if n <= q2 else "cabeza")
pc["grupo_frecuencia"] = pc.n_cajas_train.apply(grp)
save_csv(pc, SUB, "ap_vs_frecuencia_entrenamiento")
gtab = pc.groupby(["split", "grupo_frecuencia"]).agg(mAP50=("AP50", "mean"), mAP50_95=("AP50_95", "mean"), n_especies=("cls", "count")).reset_index()
save_csv(gtab, SUB, "map_por_grupo_frecuencia"); print(gtab.round(3).to_string(index=False))
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, s in zip(axs, ["cis_test", "trans_test"]):
    d = pc[pc.split == s]
    ax.scatter(d.n_cajas_train.clip(lower=1), d.AP50, s=30)
    for r in d.itertuples(): ax.annotate(r.especie, (max(r.n_cajas_train, 1), r.AP50), fontsize=7, xytext=(3, 2), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("cajas de entrenamiento (log)"); ax.set_title(s)
    if len(d) >= 3:
        rho, pv = stats.spearmanr(d.n_cajas_train, d.AP50); ax.text(0.02, 0.95, f"Spearman ρ={rho:.2f} (p={pv:.3f})", transform=ax.transAxes, fontsize=8)
        log_result("desbalance", f"spearman_{s}", dict(rho=rho, p=pv))
axs[0].set_ylabel("AP@0.5"); fig.suptitle("AP por especie vs. frecuencia en entrenamiento")
save_fig(fig, SUB, "ap_vs_frecuencia")
log_result("desbalance", "umbrales_terciles_cajas", [float(q1), float(q2)])
log_result("desbalance", "map50_por_grupo", {f"{r.split}/{r.grupo_frecuencia}": r.mAP50 for r in gtab.itertuples()})
log_result("desbalance", "ratio_max_min_cajas_train", float(tc[tc > 0].max() / tc[tc > 0].min()))

## 10. Generalización entre ubicaciones de cámara (cis vs trans)
Se reporta la brecha global y por especie, y se separa **localización** (detectar "un animal" sin importar la especie, con NMS agnóstica a la clase) de **clasificación** (acierto de especie dado que la caja se localizó). También se mide la tasa de falsos positivos en imágenes vacías de cámaras vistas vs. nuevas.

In [ ]:
#@title 10. Brecha cis → trans
SUB = "04_generalizacion"
s_cis, s_trans = summary_df.set_index("split").loc["cis_test"], summary_df.set_index("split").loc["trans_test"]
gap = pd.DataFrame({m: [s_cis[m], s_trans[m], s_trans[m] - s_cis[m], 100 * (s_trans[m] - s_cis[m]) / max(s_cis[m], 1e-9)]
                    for m in ["mAP50", "mAP50_95", "P", "R"]}, index=["cis_test (vistas)", "trans_test (nuevas)", "diferencia", "diferencia_%"]).T
save_csv(gap.reset_index().rename(columns={"index": "métrica"}), SUB, "brecha_global"); print(gap.round(4))
for m in ["mAP50", "mAP50_95"]: log_result("generalizacion", f"brecha_{m}", float(gap.loc[m, "diferencia"]))

pcc = per_class_df.pivot_table(index="especie", columns="split", values="AP50")
pcc = pcc[[c for c in ["cis_test", "trans_test"] if c in pcc]].dropna(how="all")
if {"cis_test", "trans_test"} <= set(pcc.columns):
    pcc["brecha"] = pcc.trans_test - pcc.cis_test; pcc = pcc.sort_values("cis_test", ascending=False)
save_csv(pcc.reset_index(), SUB, "ap50_por_especie_cis_vs_trans")
fig, ax = plt.subplots(figsize=(9, 3.5)); x = np.arange(len(pcc))
ax.bar(x - 0.2, pcc.get("cis_test", 0), 0.4, label="cis_test (cámaras vistas)"); ax.bar(x + 0.2, pcc.get("trans_test", 0), 0.4, label="trans_test (cámaras nuevas)")
ax.set_xticks(x); ax.set_xticklabels(pcc.index, rotation=45, ha="right"); ax.set_ylabel("AP@0.5"); ax.legend(); ax.set_title("AP por especie: ubicaciones vistas vs. nuevas")
save_fig(fig, SUB, "ap50_por_especie_cis_vs_trans")

def agnostic_nms(p, iou=0.7):
    import torchvision
    out = []
    for k, d in p[p.conf >= EVAL_MIN_CONF].groupby("key"):
        keep = torchvision.ops.nms(torch.tensor(d[["x1", "y1", "x2", "y2"]].values, dtype=torch.float32), torch.tensor(d.conf.values, dtype=torch.float32), iou).numpy()
        out.append(d.iloc[keep])
    return pd.concat(out, ignore_index=True) if out else p.iloc[:0]

def loc_cls_analysis(split, conf_thr):
    g = gt_of(split); p = agnostic_nms(preds[split]); p = p[p.conf >= conf_thr]
    pg = {k: v for k, v in p.groupby("key")}; rows = []
    for k, gg in g.groupby("key"):
        dd = pg.get(k)
        for r in gg.itertuples():
            loc_ok, cls_ok = False, False
            if dd is not None:
                ious = iou_matrix(np.array([[r.x1, r.y1, r.x2, r.y2]]), dd[["x1", "y1", "x2", "y2"]].values)[0]
                cand = np.where(ious >= 0.5)[0]
                if len(cand):
                    j = cand[np.argmax(dd.conf.values[cand])]; loc_ok = True; cls_ok = int(dd.cls.values[j]) == r.cls
            rows.append(dict(key=k, location=r.location, cls=r.cls, localizado=loc_ok, especie_correcta=cls_ok))
    return pd.DataFrame(rows)

lc = pd.concat([loc_cls_analysis(s, CONF_OP).assign(split=s) for s in ["cis_test", "trans_test"]], ignore_index=True)
lc_s = lc.groupby("split").apply(lambda d: pd.Series({"recall_localizacion": d.localizado.mean(),
        "acc_especie_dado_localizado": d[d.localizado].especie_correcta.mean(), "recall_con_especie": d.especie_correcta.mean(), "n_gt": len(d)})).reset_index()
save_csv(lc_s, SUB, "localizacion_vs_clasificacion"); print(lc_s.round(3).to_string(index=False))
for r in lc_s.itertuples():
    log_result("generalizacion", f"{r.split}_recall_localizacion", r.recall_localizacion)
    log_result("generalizacion", f"{r.split}_acc_especie_dado_localizado", r.acc_especie_dado_localizado)
lc_loc = lc.groupby(["split", "location"]).agg(n_gt=("cls", "size"), recall_localizacion=("localizado", "mean"),
        acc_especie=("especie_correcta", lambda s: s[lc.loc[s.index, "localizado"]].mean())).reset_index()
save_csv(lc_loc, SUB, "por_ubicacion")
fig, ax = plt.subplots(figsize=(9, 3.5))
lc_loc = lc_loc.sort_values(["split", "acc_especie"]); colors = {"cis_test": "tab:blue", "trans_test": "tab:orange"}
ax.bar(range(len(lc_loc)), lc_loc.acc_especie, color=[colors[s] for s in lc_loc.split])
ax.set_xticks(range(len(lc_loc))); ax.set_xticklabels(lc_loc.location, rotation=90, fontsize=7)
ax.set_ylabel("exactitud de especie | localizado"); ax.set_xlabel("ubicación de cámara (azul=vista, naranja=nueva)")
ax.set_title(f"Exactitud de especie por cámara (conf ≥ {CONF_OP})"); save_fig(fig, SUB, "exactitud_especie_por_camara")

# Falsos positivos en vacías
fp_rows = []
for s in ["cis_test", "trans_test"]:
    ek = set(data[(data.split == s) & (data.status == "vacia")].key)
    if not ek: continue
    hit = set(preds[s][(preds[s].conf >= CONF_OP) & preds[s].key.isin(ek)].key)
    fp_rows.append(dict(split=s, n_vacias=len(ek), pct_vacias_con_falso_positivo=100 * len(hit) / len(ek)))
fp_df = pd.DataFrame(fp_rows); save_csv(fp_df, SUB, "falsos_positivos_en_vacias"); print(fp_df)
for r in fp_df.itertuples(): log_result("generalizacion", f"{r.split}_pct_vacias_con_FP", r.pct_vacias_con_falso_positivo)

## 11. ¿Aprende el animal o el fondo? Grad-CAM + pruebas de fondo
Tres evidencias complementarias:
1. **Grad-CAM** sobre capas del backbone de YOLOv8 (`GRADCAM_LAYERS`), con objetivo = puntaje máximo de la clase verdadera. Se mide la **fracción de energía del mapa dentro de la caja GT** y su *lift* (= fracción / área relativa de la caja; 1 = no mejor que el azar). Si el modelo usa el fondo, la energía dentro de la caja baja, sobre todo en cámaras nuevas.
2. **Intercambio de fondo**: el recorte del animal se pega sobre una imagen vacía de (a) la misma cámara y (b) una cámara distinta (vista vs. nueva). Si el desempeño depende del fondo, cambia aunque el animal sea idéntico.
3. **Animal borrado**: se rellena la caja con gris; si el modelo sigue prediciendo la especie con confianza, la decisión se apoya en el contexto.

In [ ]:
#@title 11a. Grad-CAM para YOLOv8 (implementación propia con hooks)
SUB = "05_gradcam"

def letterbox(img, S=IMGSZ):
    h, w = img.shape[:2]; r = min(S / h, S / w); nh, nw = round(h * r), round(w * r)
    top, left = (S - nh) // 2, (S - nw) // 2
    out = np.full((S, S, 3), 114, np.uint8); out[top:top + nh, left:left + nw] = cv2.resize(img, (nw, nh))
    return out, (top, left, nh, nw)

class YoloGradCAM:
    def __init__(self, weights, layers):
        self.net = YOLO(str(weights)).model.float().to("cuda" if torch.cuda.is_available() else "cpu").eval()
        for p in self.net.parameters(): p.requires_grad_(True)
        self.dev = next(self.net.parameters()).device; self.layers = layers; self.acts, self.grads = {}, {}
        for li in layers: self.net.model[li].register_forward_hook(self._hook(li))
    def _hook(self, li):
        def f(m, i, o):
            self.acts[li] = o
            o.register_hook(lambda g: self.grads.__setitem__(li, g))
        return f
    def __call__(self, img_bgr, cls):
        lb, (top, left, nh, nw) = letterbox(img_bgr)
        x = torch.from_numpy(lb[..., ::-1].copy()).permute(2, 0, 1)[None].float().div(255).to(self.dev)
        self.net.zero_grad(); out = self.net(x); y = out[0] if isinstance(out, (list, tuple)) else out
        scores = y[0, 4 + cls]; a = int(scores.argmax()); score = scores[a]; score.backward()
        box = y[0, :4, a].detach().cpu().numpy()  # xywh en píxeles del letterbox
        cams = {}
        for li in self.layers:
            A, G = self.acts[li][0], self.grads[li][0]
            cam = torch.relu((G.mean(dim=(1, 2), keepdim=True) * A).sum(0)).detach()
            cam = torch.nn.functional.interpolate(cam[None, None], size=lb.shape[:2], mode="bilinear", align_corners=False)[0, 0]
            cam = cam[top:top + nh, left:left + nw].cpu().numpy()
            cams[li] = cv2.resize(cam, (img_bgr.shape[1], img_bgr.shape[0]))
        h, w = img_bgr.shape[:2]; r = nh / h
        cx, cy, bw, bh = box; pb = np.array([(cx - bw / 2 - left) / r / w, (cy - bh / 2 - top) / r / h, (cx + bw / 2 - left) / r / w, (cy + bh / 2 - top) / r / h])
        return cams, float(score.detach()), pb

def energy_in_box(cam, b):
    h, w = cam.shape; tot = cam.sum() + 1e-12
    x1, y1, x2, y2 = int(b[0] * w), int(b[1] * h), max(int(b[0] * w) + 1, int(b[2] * w)), max(int(b[1] * h) + 1, int(b[3] * h))
    frac = cam[y1:y2, x1:x2].sum() / tot; area = (b[2] - b[0]) * (b[3] - b[1])
    return float(frac), float(frac / max(area, 1e-6))

gc = YoloGradCAM(BEST, GRADCAM_LAYERS)
op_pred = {s: preds[s][preds[s].conf >= CONF_OP] for s in ["cis_test", "trans_test"]}
rows, examples = [], []
rng = np.random.default_rng(SEED)
for s in ["cis_test", "trans_test"]:
    one = gt_of(s).groupby("key").filter(lambda d: len(d) == 1)
    one = one.sample(min(N_GRADCAM, len(one)), random_state=SEED)
    pk = {k: v for k, v in op_pred[s].groupby("key")}
    for r in tqdm(one.itertuples(), total=len(one), desc=f"Grad-CAM {s}"):
        img = cv2.imread(str(YOLO_DIR / "images" / s / f"{r.key}.jpg")); b = np.array([r.x1, r.y1, r.x2, r.y2])
        cams, score, pb = gc(img, int(r.cls))
        d = pk.get(r.key); ok = False
        if d is not None:
            dd = d[d.cls == r.cls]
            ok = len(dd) > 0 and iou_matrix(b[None], dd[["x1", "y1", "x2", "y2"]].values).max() >= 0.5
        row = dict(split=s, key=r.key, cls=r.cls, especie=NAMES[r.cls], location=r.location, detectado_correcto=bool(ok), score_clase=score,
                   cam_nulo=bool(all(c.max() <= 0 for c in cams.values())),
                   area_rel=float((b[2] - b[0]) * (b[3] - b[1])), iou_ancla_top=float(iou_matrix(b[None], pb[None])[0, 0]))
        for li, cam in cams.items():
            f, lift = energy_in_box(cam, b); row[f"energia_caja_L{li}"] = f; row[f"lift_L{li}"] = lift
        rows.append(row)
        if len(examples) < 400: examples.append((s, r.key, int(r.cls), b, cams[GRADCAM_LAYERS[-1]], ok))
gcdf = pd.DataFrame(rows); save_csv(gcdf, SUB, "gradcam_por_imagen")
agg = gcdf.groupby(["split", "detectado_correcto"])[[c for c in gcdf if c.startswith(("energia", "lift"))]].median().reset_index()
save_csv(agg, SUB, "gradcam_resumen_medianas"); print(agg.round(3).to_string(index=False))
for li in GRADCAM_LAYERS:
    a_ = gcdf[gcdf.split == "cis_test"][f"energia_caja_L{li}"]; b_ = gcdf[gcdf.split == "trans_test"][f"energia_caja_L{li}"]
    if len(a_) > 3 and len(b_) > 3:
        u, pv = stats.mannwhitneyu(a_, b_)
        log_result("gradcam", f"L{li}_energia_mediana", dict(cis=float(a_.median()), trans=float(b_.median()), mannwhitney_p=float(pv)))
    log_result("gradcam", f"L{li}_lift_mediano", gcdf.groupby("split")[f"lift_L{li}"].median().to_dict())
log_result("gradcam", "pct_mapas_nulos", float(100 * gcdf.cam_nulo.mean()))

fig, axs = plt.subplots(1, len(GRADCAM_LAYERS), figsize=(5 * len(GRADCAM_LAYERS), 3.2))
axs = np.atleast_1d(axs)
for ax, li in zip(axs, GRADCAM_LAYERS):
    groups = [(f"{s}\n{'acierto' if ok else 'fallo'}", gcdf[(gcdf.split == s) & (gcdf.detectado_correcto == ok)][f"energia_caja_L{li}"])
              for s in ["cis_test", "trans_test"] for ok in [True, False]]
    groups = [(n, v) for n, v in groups if len(v)]
    ax.boxplot([v.values for _, v in groups], showfliers=False)
    ax.set_xticks(range(1, len(groups) + 1)); ax.set_xticklabels([n for n, _ in groups], fontsize=7)
    ax.set_title(f"Energía Grad-CAM dentro de la caja GT (capa {li})"); ax.set_ylabel("fracción de energía")
save_fig(fig, SUB, "energia_gradcam_en_caja")

def show_grid(sel, title, name):
    n = len(sel)
    if n == 0: return
    fig, axs = plt.subplots(2, n, figsize=(2.6 * n, 4.6)); axs = np.array(axs).reshape(2, n)
    for j, (s, k, c, b, cam, ok) in enumerate(sel):
        img = cv2.imread(str(YOLO_DIR / "images" / s / f"{k}.jpg"))
        axs[0, j].imshow(draw_boxes(img.copy(), [b], (0, 255, 0))[..., ::-1]); axs[0, j].set_title(f"{NAMES[c]} · {'✔' if ok else '✘'}", fontsize=8)
        heat = cv2.applyColorMap((255 * cam / (cam.max() + 1e-12)).astype(np.uint8), cv2.COLORMAP_JET)
        axs[1, j].imshow(draw_boxes(cv2.addWeighted(img, 0.5, heat, 0.5, 0), [b], (255, 255, 255))[..., ::-1])
        for a in axs[:, j]: a.axis("off")
    fig.suptitle(title, fontsize=10); save_fig(fig, SUB, name)
for s in ["cis_test", "trans_test"]:
    for ok in [True, False]:
        sel = [e for e in examples if e[0] == s and e[5] == ok][:5]
        show_grid(sel, f"Grad-CAM capa {GRADCAM_LAYERS[-1]} · {s} · {'aciertos' if ok else 'fallos'} (verde = caja GT)", f"gradcam_{s}_{'aciertos' if ok else 'fallos'}")
# un ejemplo por especie (para el informe)
per_sp = []
for c in range(NC):
    e = [x for x in examples if x[2] == c]
    if e: per_sp.append(e[0])
for i in range(0, len(per_sp), 6): show_grid(per_sp[i:i + 6], "Grad-CAM: un ejemplo por especie", f"gradcam_por_especie_{i // 6}")

In [ ]:
#@title 11b. Intercambio de fondo y animal borrado
empties = data[data.status == "vacia"]
emp_by_loc = {l: d.key.tolist() for l, d in empties.groupby("location")}
emp_path = dict(zip(empties.key, empties.yolo_path))
seen_locs = set(data.loc[data.split == "train", "location"]); unseen_locs = set(data.loc[data.split == "trans_test", "location"])

def run_det(imgs_list):
    res = model.predict(imgs_list, imgsz=IMGSZ, conf=0.01, iou=0.7, device=DEVICE, verbose=False, batch=len(imgs_list))
    return [(r.boxes.xyxyn.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)) for r in res]

def evaluate_case(det, b, c):
    xy, cf, cl = det
    m = cl == c
    if not m.any(): return 0.0, 0.0, False
    ious = iou_matrix(b[None], xy[m])[0]; conf_any = float(cf[m].max())
    conf_loc = float(cf[m][ious >= 0.5].max()) if (ious >= 0.5).any() else 0.0
    return conf_loc, conf_any, conf_loc >= CONF_OP

swap_rows = []
directions = [("cis_test", "misma cámara (vista)", "cámara nueva"), ("trans_test", "misma cámara (nueva)", "cámara vista")]
for s, lab_same, lab_other in directions:
    one = gt_of(s).groupby("key").filter(lambda d: len(d) == 1)
    one = one[one.location.isin(emp_by_loc.keys())]
    one = one.sample(min(N_BGSWAP, len(one)), random_state=SEED)
    other_pool = [k for l in (unseen_locs if s == "cis_test" else seen_locs) for k in emp_by_loc.get(l, [])]
    if len(one) == 0 or not other_pool:
        print(f"[{s}] no hay suficientes vacías para la prueba de fondo; se omite."); continue
    for r in tqdm(one.itertuples(), total=len(one), desc=f"fondos {s}"):
        src = cv2.imread(str(YOLO_DIR / "images" / s / f"{r.key}.jpg")); h, w = src.shape[:2]
        b = np.array([r.x1, r.y1, r.x2, r.y2]); x1, y1, x2, y2 = int(b[0] * w), int(b[1] * h), max(int(b[0] * w) + 1, int(b[2] * w)), max(int(b[1] * h) + 1, int(b[3] * h))
        same_c = [k for k in emp_by_loc[r.location] if k != r.key]
        if not same_c: continue
        bg_same = cv2.resize(cv2.imread(emp_path[rng.choice(same_c)]), (w, h)); bg_other = cv2.resize(cv2.imread(emp_path[rng.choice(other_pool)]), (w, h))
        comp_same, comp_other = bg_same.copy(), bg_other.copy()
        comp_same[y1:y2, x1:x2] = src[y1:y2, x1:x2]; comp_other[y1:y2, x1:x2] = src[y1:y2, x1:x2]
        erased = src.copy(); erased[y1:y2, x1:x2] = 114
        dets = run_det([src, comp_same, comp_other, erased])
        for cond, det in zip(["original", lab_same, lab_other, "animal borrado"], dets):
            cl_, ca_, hit = evaluate_case(det, b, int(r.cls))
            swap_rows.append(dict(split=s, key=r.key, especie=NAMES[int(r.cls)], condicion=cond, conf_clase_en_caja=cl_, conf_clase_max=ca_, detectado=hit))
        if len([x for x in swap_rows if x["split"] == s and x["condicion"] == "original"]) <= 4:
            fig, axs = plt.subplots(1, 4, figsize=(12, 2.6))
            for ax, im_, t in zip(axs, [src, comp_same, comp_other, erased], ["original", lab_same, lab_other, "animal borrado"]):
                ax.imshow(im_[..., ::-1]); ax.set_title(t, fontsize=8); ax.axis("off")
            save_fig(fig, SUB, f"ejemplo_intercambio_fondo_{s}_{r.key[:20]}")
sw = pd.DataFrame(swap_rows)
if len(sw):
    save_csv(sw, SUB, "intercambio_fondo_por_ejemplo")
    swt = sw.groupby(["split", "condicion"]).agg(tasa_deteccion=("detectado", "mean"), conf_media_en_caja=("conf_clase_en_caja", "mean"),
                                               conf_media_max=("conf_clase_max", "mean"), n=("key", "count")).reset_index()
    save_csv(swt, SUB, "intercambio_fondo_resumen"); print(swt.round(3).to_string(index=False))
    for s, lab_same, lab_other in directions:
        d = sw[sw.split == s]
        if d.empty: continue
        a_ = d[d.condicion == lab_same].set_index("key").conf_clase_en_caja; b_ = d[d.condicion == lab_other].set_index("key").conf_clase_en_caja
        common = a_.index.intersection(b_.index)
        if len(common) > 5:
            try: pv = stats.wilcoxon(a_[common], b_[common]).pvalue
            except ValueError: pv = np.nan
            log_result("fondo", f"{s}_wilcoxon_p_misma_vs_otra", float(pv))
        log_result("fondo", f"{s}_tasa_deteccion", d.groupby("condicion").detectado.mean().to_dict())
        log_result("fondo", f"{s}_conf_clase_max_animal_borrado", float(d[d.condicion == "animal borrado"].conf_clase_max.mean()))
    fig, ax = plt.subplots(figsize=(8, 3.2))
    piv = swt.pivot(index="condicion", columns="split", values="tasa_deteccion")
    piv.plot.bar(ax=ax, rot=20); ax.set_ylabel(f"tasa de detección correcta (conf ≥ {CONF_OP})"); ax.set_title("Mismo animal, distinto fondo")
    save_fig(fig, SUB, "intercambio_fondo")

## 12. Robustez ante oclusión y tamaño del objeto
CCT no anota oclusión, así que se **estima** con un proxy reproducible por caja GT:
* **solapamiento**: máxima fracción de la caja cubierta por otra caja GT (oclusión entre animales);
* **truncamiento**: la caja toca el borde de la imagen (el animal sale del encuadre);
* **desorden/vegetación**: densidad de bordes dentro de la caja, como percentil dentro de su especie (normaliza la textura propia del animal).

`score = 0.4·solapamiento + 0.3·truncamiento + 0.3·desorden` → terciles = **bajo / medio / alto**. Se exporta una muestra para **validación manual** (ver 12b). El tamaño se agrupa por terciles del **área relativa** de la caja. La evaluación por subgrupo usa el protocolo COCO de regiones ignoradas (las cajas fuera del subgrupo se ignoran; para tamaño también se ignoran detecciones no emparejadas fuera del rango).

In [ ]:
#@title 12. Partición por oclusión estimada, tamaño y día/noche
SUB = "06_robustez"
T = gt[gt.split.isin(["cis_test", "trans_test"])].copy()
T["rel_area"] = (T.x2 - T.x1) * (T.y2 - T.y1)
def max_ioa(d):
    b = d[["x1", "y1", "x2", "y2"]].values
    if len(b) == 1: return pd.Series([0.0], index=d.index)
    ix1 = np.maximum(b[:, None, 0], b[None, :, 0]); iy1 = np.maximum(b[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(b[:, None, 2], b[None, :, 2]); iy2 = np.minimum(b[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None); np.fill_diagonal(inter, 0)
    return pd.Series((inter / ((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]))[:, None]).max(1), index=d.index)
T["solapamiento"] = T.groupby("key", group_keys=False).apply(max_ioa).clip(0, 1)
eps = 0.01
T["truncamiento"] = (((T.x1 <= eps).astype(int) + (T.y1 <= eps).astype(int) + (T.x2 >= 1 - eps).astype(int) + (T.y2 >= 1 - eps).astype(int)) / 2).clip(0, 1)
T["desorden"] = T.groupby("cls").edge_density.rank(pct=True)
T["score_oclusion"] = 0.4 * T.solapamiento + 0.3 * T.truncamiento + 0.3 * T.desorden
oq = np.quantile(T.score_oclusion, [1 / 3, 2 / 3]); aq = np.quantile(T.rel_area, [1 / 3, 2 / 3])
T["oclusion"] = pd.cut(T.score_oclusion, [-np.inf, oq[0], oq[1], np.inf], labels=["bajo", "medio", "alto"]).astype(str)
T["tamano"] = pd.cut(T.rel_area, [-np.inf, aq[0], aq[1], np.inf], labels=["pequeño", "mediano", "grande"]).astype(str)
night = dict(zip(data.key, data.is_night)); T["iluminacion"] = T.key.map(night).map({True: "noche (IR)", False: "día"})
save_csv(T, SUB, "cajas_test_con_factores")
log_result("robustez", "umbrales_score_oclusion", [float(x) for x in oq]); log_result("robustez", "umbrales_area_relativa", [float(x) for x in aq])

rob = []
for s in ["cis_test", "trans_test"]:
    g = T[T.split == s].reset_index(drop=True); p = preds[s]
    parea = (p.x2 - p.x1) * (p.y2 - p.y1)
    for factor, levels in [("oclusion", ["bajo", "medio", "alto"]), ("tamano", ["pequeño", "mediano", "grande"]), ("iluminacion", ["día", "noche (IR)"])]:
        for lv in levels:
            ign = (g[factor] != lv).values
            dign = None
            if factor == "tamano":
                lo, hi = {"pequeño": (-np.inf, aq[0]), "mediano": (aq[0], aq[1]), "grande": (aq[1], np.inf)}[lv]
                dign = ~((parea > lo) & (parea <= hi)).values
            sm, _ = evaluate_det(g, p, gt_ignore=ign, det_ignore=dign, min_gt=5, conf_recall=CONF_OP)
            rob.append(dict(split=s, factor=factor, nivel=lv, **sm))
rob = pd.DataFrame(rob); save_csv(rob, SUB, "desempeno_por_subgrupo"); print(rob.round(3).to_string(index=False))
for r in rob.itertuples(): log_result("robustez", f"{r.split}/{r.factor}/{r.nivel}", dict(mAP50=r.mAP50, mAP50_95=r.mAP50_95, recall=r.recall, n_gt=r.n_gt))
fig, axs = plt.subplots(1, 3, figsize=(13, 3.3))
for ax, (factor, levels) in zip(axs, [("oclusion", ["bajo", "medio", "alto"]), ("tamano", ["pequeño", "mediano", "grande"]), ("iluminacion", ["día", "noche (IR)"])]):
    d = rob[rob.factor == factor]; x = np.arange(len(levels))
    for i, s in enumerate(["cis_test", "trans_test"]):
        v = d[d.split == s].set_index("nivel").reindex(levels)
        ax.bar(x + (i - 0.5) * 0.38, v.mAP50, 0.38, label=s)
    ax.set_xticks(x); ax.set_xticklabels(levels); ax.set_title(f"mAP@0.5 por {factor}"); ax.set_ylim(0, 1)
axs[0].legend(fontsize=7); save_fig(fig, SUB, "map50_por_oclusion_tamano_iluminacion")
fig, ax = plt.subplots(figsize=(5, 3))
for s in ["cis_test", "trans_test"]:
    d = rob[(rob.factor == "oclusion") & (rob.split == s)].set_index("nivel").reindex(["bajo", "medio", "alto"]); ax.plot(d.index, d.recall, marker="o", label=s)
ax.set_ylabel(f"recall (conf ≥ {CONF_OP})"); ax.set_title("Recall vs. oclusión estimada"); ax.legend(); save_fig(fig, SUB, "recall_vs_oclusion")

In [ ]:
#@title 12b. Muestra para validación manual del proxy de oclusión
MANUAL_IN = OUT / "06_robustez" / "oclusion_manual_etiquetada.csv"   # súbelo aquí (mismas columnas + 'nivel_manual') y re-ejecuta esta celda
samp = pd.concat([T[T.oclusion == lv].sample(min(20, (T.oclusion == lv).sum()), random_state=SEED) for lv in ["bajo", "medio", "alto"]]).sample(frac=1, random_state=SEED).reset_index(drop=True)
samp["id_muestra"] = range(1, len(samp) + 1); samp["nivel_manual"] = ""
save_csv(samp[["id_muestra", "key", "split", "cls", "x1", "y1", "x2", "y2", "oclusion", "score_oclusion", "nivel_manual"]].assign(especie=samp.cls.map(NAMES)), SUB, "oclusion_muestra_para_etiquetar")
cols = 10; rows_ = int(np.ceil(len(samp) / cols)); fig, axs = plt.subplots(rows_, cols, figsize=(2 * cols, 2.1 * rows_))
for ax in np.atleast_1d(axs).flat: ax.axis("off")
for ax, r in zip(np.atleast_1d(axs).flat, samp.itertuples()):
    im = cv2.imread(str(YOLO_DIR / "images" / r.split / f"{r.key}.jpg")); h, w = im.shape[:2]
    x1, y1, x2, y2 = int(r.x1 * w), int(r.y1 * h), int(r.x2 * w), int(r.y2 * h); pad = 15
    crop = im[max(0, y1 - pad):min(h, y2 + pad), max(0, x1 - pad):min(w, x2 + pad)]
    if crop.size: ax.imshow(crop[..., ::-1])
    ax.set_title(f"#{r.id_muestra}", fontsize=8)
fig.suptitle("Muestra ciega para etiquetar oclusión manualmente (bajo/medio/alto)"); save_fig(fig, SUB, "oclusion_muestra_para_etiquetar")
if MANUAL_IN.exists():
    from sklearn.metrics import cohen_kappa_score, confusion_matrix
    man = pd.read_csv(MANUAL_IN); man = man[man.nivel_manual.isin(["bajo", "medio", "alto"])]
    k = cohen_kappa_score(man.oclusion, man.nivel_manual, labels=["bajo", "medio", "alto"], weights="linear")
    log_result("robustez", "kappa_ponderado_proxy_vs_manual", float(k)); print("Kappa ponderado proxy vs manual:", round(k, 3))
    save_csv(pd.DataFrame(confusion_matrix(man.nivel_manual, man.oclusion, labels=["bajo", "medio", "alto"]),
             index=["man_bajo", "man_medio", "man_alto"], columns=["proxy_bajo", "proxy_medio", "proxy_alto"]), SUB, "oclusion_matriz_proxy_vs_manual", index=True)
else:
    print("Para validar el proxy: etiqueta 'nivel_manual' en oclusion_muestra_para_etiquetar.csv viendo la figura, guárdalo como", MANUAL_IN.name)

## 13. Módulo de monitoreo ecológico (basado en reglas)
Reglas, en el orden en que se aplican:
1. **Detección válida**: confianza ≥ umbral operativo (máximo F1 en validación).
2. **Foto con fauna / vacía**: una foto es "con fauna" si tiene ≥1 detección válida.
3. **Secuencia** (ráfaga de la cámara, `seq_id`): la especie se registra con su **MaxN** = máximo número de individuos en un mismo fotograma (estándar en ecología para no contar dos veces al mismo individuo).
4. **Evento independiente**: detecciones de la misma especie en la misma cámara separadas por menos de `INDEP_MINUTES` minutos se fusionan en un solo evento.
5. **Índice de abundancia relativa (RAI)** = eventos independientes / días-cámara × 100.

Se aplica a una cámara **nunca vista** (de `trans_test`) y se compara contra el mismo reporte calculado con las etiquetas verdaderas.

In [ ]:
#@title 13. Implementación y demostración
SUB = "07_monitoreo"

class MonitorEcologico:
    def __init__(self, conf_thr, indep_min=INDEP_MINUTES, period=REPORT_PERIOD, names=NAMES):
        self.conf_thr, self.indep, self.period, self.names = conf_thr, pd.Timedelta(minutes=indep_min), period, names
    def reporte(self, frames, dets):
        """frames: key, location, datetime, seq_id | dets: key, cls, conf (una fila por caja)"""
        fr = frames[["key", "location", "datetime", "seq_id"]].copy()
        d = dets[dets.conf >= self.conf_thr].merge(fr, on="key", how="inner"); d["especie"] = d.cls.map(self.names)
        per_frame = d.groupby(["key", "especie"]).size().rename("n_ind").reset_index().merge(fr, on="key")
        fr["con_fauna"] = fr.key.isin(per_frame.key)
        seq = per_frame.groupby(["location", "seq_id", "especie"]).agg(MaxN=("n_ind", "max"), t0=("datetime", "min"), t1=("datetime", "max"), n_fotos=("key", "nunique")).reset_index()
        seq = seq.sort_values(["location", "especie", "t0"])
        ev_ids, prev = [], {}
        for r in seq.itertuples():
            k = (r.location, r.especie); last = prev.get(k)
            if last is None or pd.isna(r.t0) or pd.isna(last[1]) or (r.t0 - last[1]) > self.indep:
                eid = (last[0] + 1) if last else 0
            else:
                eid = last[0]
            prev[k] = (eid, r.t1 if pd.notna(r.t1) else (last[1] if last else r.t1)); ev_ids.append(eid)
        seq["evento"] = ev_ids
        ev = seq.groupby(["location", "especie", "evento"]).agg(t0=("t0", "min"), t1=("t1", "max"), MaxN=("MaxN", "max"), n_secuencias=("seq_id", "nunique")).reset_index()
        ev["t0"] = pd.to_datetime(ev.t0); ev["t1"] = pd.to_datetime(ev.t1); d["datetime"] = pd.to_datetime(d.datetime)
        dates = fr.datetime.dropna()
        dias = max(1, (dates.max().normalize() - dates.min().normalize()).days + 1) if len(dates) else 1
        por_especie = ev.groupby("especie").agg(eventos_independientes=("evento", "count"), MaxN=("MaxN", "max")).join(
            d.groupby("especie").agg(detecciones=("cls", "size"), fotos=("key", "nunique"), conf_media=("conf", "mean"))).join(
            seq.groupby("especie").size().rename("secuencias")).fillna(0)
        por_especie["RAI_eventos_x100_dias"] = 100 * por_especie.eventos_independientes / dias
        ev["periodo"] = ev.t0.dt.to_period(self.period).astype(str)
        d["periodo"] = d.datetime.dt.to_period(self.period).astype(str)
        por_periodo = ev.groupby(["periodo", "especie"]).agg(eventos=("evento", "count"), MaxN=("MaxN", "max")).join(
            d.groupby(["periodo", "especie"]).agg(detecciones=("cls", "size"), fotos=("key", "nunique"))).fillna(0).reset_index()
        ev["hora"] = ev.t0.dt.hour
        actividad_horaria = ev.groupby(["especie", "hora"]).size().rename("eventos").reset_index()
        camara = dict(fotos=len(fr), fotos_con_fauna=int(fr.con_fauna.sum()), fotos_vacias=int((~fr.con_fauna).sum()),
                      pct_vacias=100 * (~fr.con_fauna).mean(), riqueza_especies=int(ev.especie.nunique()),
                      especies=sorted(ev.especie.unique().tolist()), dias_camara=int(dias),
                      desde=str(dates.min()) if len(dates) else None, hasta=str(dates.max()) if len(dates) else None,
                      secuencias=int(fr.seq_id.nunique()), eventos_independientes=int(len(ev)))
        return dict(camara=camara, por_especie=por_especie.reset_index(), por_periodo=por_periodo, actividad_horaria=actividad_horaria, eventos=ev)

mon = MonitorEcologico(CONF_OP)
tt = data[data.split == "trans_test"]
cand = gt_of("trans_test").groupby("location").agg(riqueza=("cls", "nunique"), cajas=("cls", "size"))
cand["fotos"] = tt.groupby("location").size(); cand = cand.sort_values(["riqueza", "fotos"], ascending=False)
DEMO_CAM = cand.index[0]; print("Cámara de demostración (no vista en entrenamiento):", DEMO_CAM); print(cand.head())
log_result("monitoreo", "camara_demo", str(DEMO_CAM))
frames = tt[tt.location == DEMO_CAM]
gt_dets = gt_of("trans_test")[["key", "cls"]].assign(conf=1.0)
rep_pred = mon.reporte(frames, preds["trans_test"]); rep_gt = mon.reporte(frames, gt_dets)
for k in ["por_especie", "por_periodo", "actividad_horaria", "eventos"]:
    save_csv(rep_pred[k], SUB, f"camara_{DEMO_CAM}_{k}_modelo"); save_csv(rep_gt[k], SUB, f"camara_{DEMO_CAM}_{k}_verdad")
cam_tab = pd.DataFrame({"modelo": rep_pred["camara"], "verdad (etiquetas)": rep_gt["camara"]})
save_csv(cam_tab.reset_index().rename(columns={"index": "indicador"}), SUB, f"camara_{DEMO_CAM}_resumen"); print(cam_tab)
cmp = rep_pred["por_especie"].set_index("especie")[["eventos_independientes", "RAI_eventos_x100_dias", "MaxN"]].join(
      rep_gt["por_especie"].set_index("especie")[["eventos_independientes", "RAI_eventos_x100_dias", "MaxN"]], lsuffix="_modelo", rsuffix="_verdad", how="outer").fillna(0)
cmp["error_eventos"] = cmp.eventos_independientes_modelo - cmp.eventos_independientes_verdad
save_csv(cmp.reset_index(), SUB, f"camara_{DEMO_CAM}_comparacion_especies"); print(cmp)
log_result("monitoreo", "resumen_camara_modelo", rep_pred["camara"]); log_result("monitoreo", "resumen_camara_verdad", rep_gt["camara"])
log_result("monitoreo", "error_absoluto_eventos_total", float(cmp.error_eventos.abs().sum()))

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
x = np.arange(len(cmp)); axs[0].bar(x - 0.2, cmp.eventos_independientes_verdad, 0.4, label="verdad"); axs[0].bar(x + 0.2, cmp.eventos_independientes_modelo, 0.4, label="modelo")
axs[0].set_xticks(x); axs[0].set_xticklabels(cmp.index, rotation=45, ha="right"); axs[0].set_ylabel("eventos independientes"); axs[0].legend(); axs[0].set_title(f"Cámara {DEMO_CAM}: eventos por especie")
for lab, rep in [("verdad", rep_gt), ("modelo", rep_pred)]:
    ah = rep["actividad_horaria"].groupby("hora").eventos.sum().reindex(range(24), fill_value=0); axs[1].plot(ah.index, ah.values, marker=".", label=lab)
axs[1].set_xlabel("hora del día"); axs[1].set_ylabel("eventos"); axs[1].set_title("Patrón de actividad diaria"); axs[1].legend()
save_fig(fig, SUB, f"camara_{DEMO_CAM}_reporte")
pp = rep_pred["por_periodo"].pivot_table(index="periodo", columns="especie", values="eventos", aggfunc="sum", fill_value=0)
if len(pp):
    fig, ax = plt.subplots(figsize=(10, 3.2)); pp.plot.bar(stacked=True, ax=ax, width=0.85); ax.set_ylabel("eventos independientes")
    ax.set_title(f"Cámara {DEMO_CAM}: actividad por periodo ({REPORT_PERIOD}) según el modelo"); ax.legend(fontsize=6, ncol=4); save_fig(fig, SUB, f"camara_{DEMO_CAM}_actividad_por_periodo")

# Todas las cámaras no vistas
allc = []
for loc, fr_ in tt.groupby("location"):
    a = mon.reporte(fr_, preds["trans_test"])["camara"]; b = mon.reporte(fr_, gt_dets)["camara"]
    allc.append(dict(camara=loc, fotos=a["fotos"], pct_vacias_modelo=a["pct_vacias"], pct_vacias_verdad=b["pct_vacias"],
                     riqueza_modelo=a["riqueza_especies"], riqueza_verdad=b["riqueza_especies"],
                     eventos_modelo=a["eventos_independientes"], eventos_verdad=b["eventos_independientes"]))
allc = pd.DataFrame(allc); save_csv(allc, SUB, "todas_camaras_no_vistas"); print(allc)
log_result("monitoreo", "camaras_no_vistas_error_medio_riqueza", float((allc.riqueza_modelo - allc.riqueza_verdad).abs().mean()))

## 14. Ablaciones (manejo de vacías y desbalance)
Todas con `ABL_EPOCHS` épocas e idéntica configuración, para comparar de forma justa:
* **base**: configuración del modelo principal (con `BG_RATIO` de vacías).
* **sin_vacias**: sin imágenes de fondo → mide el aporte de las vacías a la reducción de falsos positivos.
* **rfs**: *Repeat Factor Sampling* (sobremuestreo de imágenes con especies raras) → estrategia de mitigación del desbalance.

In [ ]:
#@title 14. Ablaciones
SUB = "08_ablacion"
if RUN_ABLATIONS:
    lists = {"base": "train_base.txt", "sin_vacias": "train_sin_vacias.txt", "rfs": "train_rfs.txt"}
    abl_rows, abl_cls = [], []
    for a in ABLATIONS:
        rd = train_run(f"abl_{a}", YOLO_DIR / lists[a], ABL_EPOCHS)
        m = YOLO(str(rd / "weights" / "best.pt"))
        for s in ["cis_test", "trans_test"]:
            sm, pcl = ul_val(m, s, run_name=f"abl_{a}"); sm.pop("_cm", None)
            ek = data[(data.split == s) & (data.status == "vacia")]
            fp = np.nan
            if len(ek):
                pe = predict_df(m, list(zip(ek.key, ek.yolo_path)), conf=CONF_OP)
                fp = 100 * pe.key.nunique() / len(ek)
            abl_rows.append(dict(ablacion=a, **sm, pct_vacias_con_FP=fp))
            abl_cls.append(pcl.assign(ablacion=a))
    abl = pd.DataFrame(abl_rows); abl_c = pd.concat(abl_cls, ignore_index=True)
    abl_c["grupo_frecuencia"] = abl_c.cls.map(train_counts).fillna(0).apply(grp)
    save_csv(abl, SUB, "ablacion_resumen"); save_csv(abl_c, SUB, "ablacion_por_especie")
    tail = abl_c.groupby(["ablacion", "split", "grupo_frecuencia"]).AP50.mean().unstack("grupo_frecuencia").reset_index()
    save_csv(tail, SUB, "ablacion_map50_por_grupo_frecuencia"); print(abl.round(4).to_string(index=False)); print(tail.round(3))
    for r in abl.itertuples(): log_result("ablacion", f"{r.ablacion}/{r.split}", dict(mAP50=r.mAP50, mAP50_95=r.mAP50_95, pct_vacias_con_FP=r.pct_vacias_con_FP))
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
    abl.pivot(index="ablacion", columns="split", values="mAP50").plot.bar(ax=axs[0], rot=0); axs[0].set_title("mAP@0.5 por ablación")
    abl.pivot(index="ablacion", columns="split", values="pct_vacias_con_FP").plot.bar(ax=axs[1], rot=0); axs[1].set_title(f"% vacías con falso positivo (conf ≥ {CONF_OP})")
    save_fig(fig, SUB, "ablacion")
else:
    print("Ablaciones desactivadas (RUN_ABLATIONS=False).")

In [ ]:
#@title 15. Velocidad de inferencia (para la discusión de despliegue)
SUB = "03_evaluacion"
d = data[data.split == "trans_test"].sample(min(200, (data.split == "trans_test").sum()), random_state=SEED)
_ = model.predict(d.yolo_path.tolist()[:8], imgsz=IMGSZ, device=DEVICE, verbose=False)
t0 = time.time()
for p in d.yolo_path: model.predict(p, imgsz=IMGSZ, device=DEVICE, verbose=False)
ms = 1000 * (time.time() - t0) / len(d)
n_params = sum(p.numel() for p in model.model.parameters())
log_result("despliegue", "ms_por_imagen_batch1", ms); log_result("despliegue", "imagenes_por_hora", 3.6e6 / ms)
log_result("despliegue", "parametros_millones", n_params / 1e6); log_result("despliegue", "dispositivo", env["gpu"])
print(f"{ms:.1f} ms/imagen (batch=1, {env['gpu']}), {n_params / 1e6:.1f} M parámetros")

## 16. Exportación de resultados
Genera `RESUMEN.md` (todas las cifras clave en un solo lugar), copia las curvas y empaqueta `outputs/` en un `.zip` en Drive y lo descarga.

In [ ]:
#@title 16. RESUMEN.md + zip
def md_table(df, n=40):
    df = df.head(n).copy()
    for c in df.select_dtypes("number"): df[c] = df[c].round(4)
    cols = list(df.columns); lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.values]
    return "\n".join(lines)
md = [f"# Resultados P3 — YOLO en cámaras trampa (CCT20)\n", f"Generado: {dt.datetime.now():%Y-%m-%d %H:%M} · modelo `{MODEL_WEIGHTS}` · imgsz {IMGSZ} · {env['gpu']} · ultralytics {ultralytics.__version__}\n",
      "## Dataset", md_table(st.reset_index()), "", "### Cajas por especie", md_table(cls_counts.reset_index()), "",
      "## Evaluación (validador Ultralytics)", md_table(summary_df), "", "### Por especie", md_table(per_class_df, 80), "",
      "## Desbalance", md_table(gtab), "", "## Generalización cis→trans", md_table(gap.reset_index().rename(columns={"index": "métrica"})), "",
      md_table(lc_s), ""]
if "fp_df" in globals() and len(fp_df): md += [md_table(fp_df), ""]
if "agg" in globals(): md += ["## Grad-CAM (medianas)", md_table(agg), ""]
if "swt" in globals(): md += ["## Intercambio de fondo", md_table(swt), ""]
md += ["## Robustez", md_table(rob, 60), "", f"## Monitoreo ecológico (cámara {DEMO_CAM})", md_table(cam_tab.reset_index().rename(columns={"index": "indicador"})), "", md_table(cmp.reset_index()), "",
       md_table(allc), ""]
if RUN_ABLATIONS and "abl" in globals(): md += ["## Ablación", md_table(abl), "", md_table(tail), ""]
md += ["## Todas las cifras (resumen.json)", "```json", json.dumps(RESULTS, indent=1, ensure_ascii=False)[:20000], "```", "", "## Archivos"]
md += [f"- `{p.relative_to(OUT)}`" for p in sorted(OUT.rglob("*")) if p.is_file() and not str(p.relative_to(OUT)).startswith("_val_runs")]
(OUT / "RESUMEN.md").write_text("\n".join(md), encoding="utf-8")
if INCLUDE_WEIGHTS_IN_ZIP: shutil.copy(BEST, od("02_entrenamiento") / "best.pt")
ts = dt.datetime.now().strftime("%Y%m%d_%H%M")
zip_path = PERSIST / f"P3_resultados_{ts}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and "_val_runs" not in p.parts: z.write(p, p.relative_to(OUT))
print(f"ZIP: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))